# **Programación para Ciencia de Datos: Parte Práctica de la Semana 1**

## Contenido del cuaderno

Cada sección se apoya en la anterior; conviene recorrerlas en orden.

- [0. Preparación del entorno y de los datos](#0-preparación-del-entorno-y-de-los-datos)
- [I. Sintaxis y Semántica de Python](#i-sintaxis-y-semántica-de-python)
- [II. Variables en Python](#ii-variables-en-python)
- [III. Funciones](#iii-funciones)
- [IV. Operadores](#iv-operadores)
- [V. Tipos de Datos Complejos](#v-tipos-de-datos-complejos)
- [VI. Estructuras de control del flujo de ejecución en Python](#vi-estructuras-de-control-del-flujo-de-ejecución-en-python)
- [VII. Comprensiones: la escritura estándar en ciencia de datos](#vii-comprensiones-la-escritura-estándar-en-ciencia-de-datos)
- [VIII. Programación Modular](#viii-programación-modular)
- [IX. Entorno reproducible y control de versiones](#ix-entorno-reproducible-y-control-de-versiones)
- [X. Excepciones](#x-excepciones)
- [XI. Librerias para Ciencia de Datos](#xi-librerias-para-ciencia-de-datos)
- [XII. Preprocesamiento de datos](#xii-preprocesamiento-de-datos)
- [XIII. Ejercicios](#xiii-ejercicios)
- [XIV. Autoevaluación](#xiv-autoevaluación)
- [XV. Del cuaderno al proyecto](#xv-del-cuaderno-al-proyecto)

---

**Convenciones que se usan aquí**

- Los bloques que empiezan con `# Demostración de error` fallan **a propósito**:
  muestran qué ocurre al cometer un error frecuente. Están protegidos con
  `try/except` para que el cuaderno pueda correr completo.
- Las citas con `>` señalan advertencias, criterios de decisión o el vínculo con
  lo que la rúbrica del proyecto evalúa.
- Todas las rutas son **relativas**: `data/raw/archivo.csv`, nunca `C:/Users/...`.

## Cómo usar este cuaderno

Este cuaderno se **ejecuta**, no se lee. Cada celda de código está pensada para
correrse, modificarse y romperse a propósito: ahí es donde se aprende.

- Ejecute las celdas **en orden**. Muchas dependen de variables creadas antes.
- Si algo falla, lea el mensaje completo: casi siempre dice qué y dónde.
- Al terminar, use **Kernel → Restart Kernel and Run All Cells**. Si el cuaderno
  corre completo, sus resultados son reproducibles; si no, encontró un problema
  que también habría aparecido en su entrega.

> **Sobre las celdas que fallan a propósito.** Varias secciones muestran errores
> típicos —nombres inválidos, objetos inmutables, argumentos que no calzan—. Están
> envueltas en `try/except` para que el cuaderno pueda correr completo, pero el
> mensaje que imprimen es exactamente el que verían al cometer ese error.

## Qué se practica aquí, y para qué

La Semana 1 cubre dos unidades temáticas, y este cuaderno sostiene los cinco
indicadores de desempeño que la Sumativa 1 evalúa sobre el proyecto:

| Contenido de este cuaderno | Indicador | Dónde se aplica en el proyecto |
| --- | --- | --- |
| Sintaxis, colecciones, control de flujo, funciones | ID 2.1 | Notebooks F1 y F2 |
| Módulos, entorno y control de versiones | ID 1.1, ID 1.2 | Repositorio y `requirements.txt` |
| Excepciones y validación | ID 2.1 | Carga y verificación de datos |
| Documentación en el notebook | ID 1.3 | Informe técnico y celdas narrativas |
| NumPy, pandas y preprocesamiento | ID 4.1 | Pipeline de la Fase 2 |

---

# 0. Preparación del entorno y de los datos

Antes de programar, dejamos el espacio de trabajo en condiciones. Esta sección
hace tres cosas: comprueba el entorno, crea la estructura de carpetas del
proyecto y genera los archivos de ejemplo que usan las secciones siguientes.

**Por qué generamos los datos en vez de descargarlos.** Así el cuaderno funciona
en cualquier computador, sin depender de archivos que puedan faltar. La
estructura de carpetas es la misma que usará el repositorio de su proyecto.

In [ ]:
import sys
import platform

# sys.version trae la versión completa con fecha de compilación;
# split()[0] deja solo el número, que es lo único que hay que comprobar.
print("Python  :", sys.version.split()[0], f"({platform.system()})")

# __import__ permite importar a partir del NOMBRE escrito como texto, algo que un
# `import numpy` normal no admite: aquí los nombres vienen de una lista.
for lib in ["numpy", "pandas", "matplotlib", "sklearn", "openpyxl"]:
    try:
        print(f"{lib:11}:", getattr(__import__(lib), "__version__", "instalado"))
    except ImportError:
        print(f"{lib:11}: NO INSTALADO  ->  python -m pip install {lib}")

# El kernel debe ser el del entorno del proyecto, no el genérico «Python 3»:
# ahí se origina la mayoría de los «a mí me funciona».
print("\nIntérprete en uso:", sys.executable)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# La semilla fija el punto de partida del generador aleatorio: con la misma
# semilla, la misma secuencia. Sin ella, los resultados cambian en cada ejecución
# y nadie puede verificarlos.
SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

# Estructura de carpetas del proyecto, con rutas RELATIVAS al cuaderno.
# Una ruta absoluta como C:/Users/... funciona en un solo computador del mundo.
for carpeta in ["data/raw", "data/processed", "src", "docs"]:
    Path(carpeta).mkdir(parents=True, exist_ok=True)

print("Estructura creada:")
for carpeta in sorted(Path(".").glob("*/")):
    if carpeta.is_dir() and not carpeta.name.startswith("."):
        print("  ", carpeta)

## 0.1 Archivos de ejemplo para la sección de lectura de datos

Las secciones sobre pandas leen archivos con separadores distintos: coma, punto
y coma, tabulación, barra vertical. Los creamos aquí para que esas celdas
funcionen, y de paso queda claro que **el separador es una decisión del archivo,
no de pandas**: por eso hay que declararlo al leer.

In [ ]:
# Cada archivo usa un separador distinto y algunas líneas de encabezado que
# habrá que saltar con skiprows. Es exactamente lo que ocurre con los datos
# públicos reales, que rara vez vienen en el formato que uno espera.
FILAS = [("Ana", 34, "Valparaiso"), ("Luis", 28, "Santiago"),
         ("Marta", 45, "Concepcion"), ("Pedro", 39, "La Serena")]

def escribir(nombre, sep, encabezado_extra=0):
    """Crea un archivo de texto con el separador indicado.

    encabezado_extra simula las líneas de metadatos que muchos archivos traen
    antes de los datos y que hay que descartar al leer.
    """
    lineas = [f"# archivo de ejemplo {nombre}"] * encabezado_extra
    lineas.append(sep.join(["nombre", "edad", "ciudad"]))
    lineas += [sep.join(map(str, f)) for f in FILAS]
    Path("data/raw", nombre).write_text("\n".join(lineas) + "\n", encoding="utf-8")

escribir("arch1.txt", ",",  encabezado_extra=2)
escribir("arch2.txt", ";",  encabezado_extra=2)
escribir("arch3.txt", "\t", encabezado_extra=3)
escribir("arch4.txt", "|",  encabezado_extra=2)
escribir("arch5.txt", ";;", encabezado_extra=2)
escribir("dat.csv",   "|")
escribir("datos.txt", "|")

# Excel necesita openpyxl; si no está instalado, el resto del cuaderno sigue igual
try:
    pd.DataFrame(FILAS, columns=["nombre", "edad", "ciudad"]).to_excel(
        "data/raw/dat_ejemplo.xlsx", sheet_name="Hoja1", index=False)
    print("Archivos de ejemplo creados, incluido el .xlsx")
except ImportError:
    print("Archivos de texto creados. Para el .xlsx: python -m pip install openpyxl")

## 0.2 El conjunto de datos de la sección de preprocesamiento

La sección IX trabaja sobre un conjunto clínico con la misma estructura del
*Stroke Prediction Dataset* de Kaggle. Si usted ya lo descargó a `data/raw/`, el
cuaderno usará ese archivo; si no, genera uno sintético equivalente.

**Los faltantes se introducen a propósito.** Un conjunto sin valores ausentes no
representa nada de lo que encontrará en la práctica, y dejaría la sección de
preprocesamiento sin objeto: no habría nada que decidir ni que justificar.

In [ ]:
ARCHIVO = Path("data/raw/healthcare-dataset-stroke-data.csv")

if ARCHIVO.exists():
    print(f"Se usará el archivo real: {ARCHIVO}")
else:
    N = 5110
    edad = np.round(rng.normal(43, 22, N).clip(0.08, 82), 1)
    hipertension = rng.binomial(1, 0.097, N)
    cardiopatia = rng.binomial(1, 0.054, N)

    # El riesgo depende de la edad y de los antecedentes: así el conjunto tiene
    # una estructura interna y no es ruido puro.
    riesgo = 0.012 + 0.13 * (edad > 65) + 0.09 * hipertension + 0.08 * cardiopatia

    sintetico = pd.DataFrame({
        "id": rng.choice(np.arange(1, 100000), size=N, replace=False),
        "gender": rng.choice(["Male", "Female", "Other"], N, p=[0.41, 0.586, 0.004]),
        "age": edad,
        "hypertension": hipertension,
        "heart_disease": cardiopatia,
        "ever_married": rng.choice(["Yes", "No"], N, p=[0.66, 0.34]),
        "work_type": rng.choice(["Private", "Self-employed", "Govt_job",
                                 "children", "Never_worked"], N,
                                p=[0.57, 0.16, 0.13, 0.13, 0.01]),
        "Residence_type": rng.choice(["Urban", "Rural"], N, p=[0.51, 0.49]),
        "avg_glucose_level": np.round(rng.gamma(6, 17, N).clip(55, 272), 2),
        "bmi": np.round(rng.normal(28.9, 7.8, N).clip(10.3, 97.6), 1),
        "smoking_status": rng.choice(
            ["never smoked", "Unknown", "formerly smoked", "smokes"],
            N, p=[0.37, 0.30, 0.17, 0.16]),
        "stroke": rng.binomial(1, riesgo.clip(0, 1)),
    })

    # replace=False evita repetir posiciones: se vacían exactamente 201 filas.
    sintetico.loc[rng.choice(N, size=201, replace=False), "bmi"] = np.nan
    sintetico.to_csv(ARCHIVO, index=False)
    print(f"Archivo sintético generado: {ARCHIVO}  ({N} filas)")
    print("Descargue el conjunto real desde Kaggle antes de usarlo en su proyecto.")

> **Comprueba.** Sin ejecutar nada, responda: si borrara la línea de la semilla,
> ¿los datos generados serían los mismos en la próxima ejecución? ¿Y en el
> computador de un compañero? Esa es la razón por la que el proyecto exige
> declararla.

# I. Sintaxis y Semántica de Python

La descripción de un lenguaje de programación viene determinada por dos componentes principales, la **sintaxis** y la **semántica** del lenguaje.
* **Sintaxis**: La sintaxis de un lenguaje de programación es el conjunto de reglas que define las combinaciones de símbolos que se consideran declaraciones o expresiones válidas en ese lenguaje.
* **Semántica**: La semántica es el campo que se ocupa del estudio matemático riguroso del significado de los lenguajes de programación. Para ello, evalúa el significado de las cadenas sintácticamente válidas definidas por un lenguaje de programación específico.

# II. Variables en Python

### 1. Variables

En el contexto de los lenguajes de programación, una variable es una entidad abstracta que permite asociar un identificador con un valor almacenado en memoria, posibilitando su referencia y modificación durante la ejecución de un programa. En Python, las variables se definen a través de una instrucción de asignación, donde un nombre simbólico queda enlazado a un objeto mediante el operador =. Este principio es clave en el desarrollo de soluciones computacionales avanzadas, ya que sustenta la representación de datos, la transformación de información y la implementación de procesos algorítmicos propios de la ciencia de datos y la inteligencia artificial.

In [ ]:
var = "Magister en Ciencia de Datos e Inteligencia Artificial"

In [ ]:
# Demostración de error: la variable no está definida en este ámbito.
# Se ejecuta dentro de try/except para que el cuaderno pueda correr
# completo; el mensaje que imprime es exactamente el que verían.
try:
    print(var)
except NameError as e:
    print(f"NameError: {e}")

En Python, el valor vinculado a una variable no es necesariamente estático, sino que puede redefinirse a lo largo de la ejecución del programa, permitiendo modelar procesos dinámicos, gestionar estados intermedios y adaptar el flujo computacional a nuevas condiciones o transformaciones de los datos.

In [ ]:
var = "Programación para Ciencia de Datos"

In [ ]:
var2 = var

In [ ]:
print(var2)

En el ejemplo presentado en las celdas anteriores, las variables var y var2 hacen referencia al mismo objeto de tipo cadena, cuyo contenido es Programación para Ciencia de Datos. En otras palabras, ambas variables apuntan a la misma instancia de texto en memoria.

### 2. Sintaxis de las variables en Python

Para definir variables en Python se debe cumplir con las reglas sintácticas definidas por el leguaje. A continuación se indican las reglas que debemos cumplir cuando definimos el nombre de una variable:

a. Los nombres de las variables en Python pueden tener cualquier longitud y pueden consistir en letras mayúsculas y minúsculas (A-Z, a-z), dígitos (0-9) y el carácter de subrayado (_)

In [ ]:
# Un nombre puede empezar con guion bajo: es válido y por convención
# indica «uso interno», algo que no se pensó para usarse desde fuera.
_var = "Ciencia de datos"

print(_var)

In [ ]:
# Python distingue mayúsculas de minúsculas: vAr, var y VAR son tres
# variables distintas. De ahí que convenga una convención estable.
vAr = "Ciencia de datos"

print(vAr)

In [ ]:
# Se pueden combinar letras, números y guion bajo, siempre que el
# nombre NO empiece con un dígito.
var_2 = "Ciencia de datos"

print(var_2)

Si una instrucción no respeta las reglas sintácticas definidas por Python, el intérprete generará un error al momento de procesar o ejecutar la sentencia correspondiente.

In [ ]:
# Demostración de error de SINTAXIS: el símbolo & no puede formar parte de un nombre.
# Un error de sintaxis impide que Python interprete la celda, así que no
# puede capturarse con try/except: el código se pasa como texto a compile().
codigo = 'var& = "Ciencia de datos"'

try:
    compile(codigo, "<demostración>", "exec")
except SyntaxError as e:
    print(f"SyntaxError: {e.msg}")
    print(f"  en el código: {codigo}")

b. Si bien los nombres de las variables en Python pueden contener dígitos, la sintaxis del lenguaje establece que el carácter inicial debe ser una letra o un guion bajo, y no un número.

In [ ]:
# Demostración de error de SINTAXIS: un nombre no puede empezar con un dígito.
# Un error de sintaxis impide que Python interprete la celda, así que no
# puede capturarse con try/except: el código se pasa como texto a compile().
codigo = '2var = "Ciencia de datos"'

try:
    compile(codigo, "<demostración>", "exec")
except SyntaxError as e:
    print(f"SyntaxError: {e.msg}")
    print(f"  en el código: {codigo}")

c. En Python, los identificadores de variables son sensibles al uso de mayúsculas y minúsculas; en consecuencia, cualquier variación en la capitalización del nombre da lugar a referencias distintas dentro del programa.

In [ ]:
Var3 = "Ciencia de datos"

In [ ]:
# Demostración de error: la variable nunca fue definida.
# Se ejecuta dentro de try/except para que el cuaderno pueda correr
# completo; el mensaje que imprime es exactamente el que verían.
try:
    print(var3)
except NameError as e:
    print(f"NameError: {e}")

### 3. Strings

Un string corresponde a una secuencia ordenada de caracteres utilizada para representar texto dentro de un programa. En Python, los valores de tipo cadena se definen delimitando los caracteres entre comillas simples (' ') o comillas dobles (" "), lo que permite almacenar y manipular información textual de manera flexible.

In [ ]:
var = "Magister en Ciencia de Datos e Inteligencia Artificial"

In [ ]:
var2 = 'Magister en Ciencia de Datos e Inteligencia Artificial'

### 4. Indexación

En diversos tipos de datos de Python es posible acceder directamente a elementos individuales dentro de una estructura ordenada mediante un índice numérico o, en otros casos, a través de una clave. Este mecanismo se conoce como indexación.

En el caso de las cadenas de texto, Python las trata como secuencias ordenadas de caracteres, por lo que cada uno de sus elementos puede ser recuperado individualmente mediante su posición. Para ello, se escribe el nombre de la variable seguido del índice correspondiente entre corchetes [].

La indexación en Python comienza en 0, lo que significa que el primer carácter de una cadena se encuentra en la posición 0, el segundo en la posición 1, y así sucesivamente. En consecuencia, el índice del último carácter corresponde a la longitud total de la cadena menos uno.

In [ ]:
nombre = "Francisco"

In [ ]:
nombre[0]

In [ ]:
nombre[4]

In [ ]:
nombre[-1]

In [ ]:
nombre[-3]

### 5. Slicing

Python también incorpora una notación específica de indexación denominada slicing, la cual permite extraer subsecuencias o subcadenas a partir de una cadena de texto. Este mecanismo resulta especialmente útil para manipular segmentos de información textual de manera eficiente y controlada.

La sintaxis general del slicing sobre una cadena s se expresa como s[m:n], donde m representa el índice inicial y n el índice final del segmento que se desea obtener. El resultado corresponde a la porción de la cadena que comienza en la posición m y se extiende hasta la posición n-1, ya que el índice superior no se incluye en la extracción.

In [ ]:
nombre = "Francisco Escudero"

In [ ]:
nombre[0:9]

In [ ]:
nombre[-12:1]

Si se omite uno de los índices en la operación de slicing, Python asume automáticamente el límite correspondiente de la secuencia; es decir, desde el inicio si no se especifica el primer valor, o hasta el final si no se indica el segundo.

In [ ]:
nombre[-9:]

In [ ]:
nombre[4:]

###

### 6. Stride

El stride es una variante avanzada del slicing que permite controlar el intervalo entre los elementos extraídos de una secuencia. En la expresión s[inicio:fin:paso], el tercer parámetro, denominado stride o paso, indica cada cuántos caracteres se seleccionará un nuevo elemento dentro de la cadena. De este modo, es posible recorrer la secuencia de forma no contigua, extrayendo caracteres según la separación especificada.

In [ ]:
nombre = "Francisco Escudero"

In [ ]:
nombre[0:8:2]

### 7. Modificación de strings

Las cadenas de texto en Python son estructuras de datos inmutables, por lo que no admiten modificaciones internas una vez instanciadas. Esto implica que no es posible reasignar directamente una porción de un string; en su lugar, debe construirse un nuevo objeto que refleje el cambio requerido.

In [ ]:
nombre = "Francisco"

In [ ]:
# Demostración de error: los strings son inmutables.
# Se ejecuta dentro de try/except para que el cuaderno pueda correr
# completo; el mensaje que imprime es exactamente el que verían.
try:
    nombre[3] = 'w'
except TypeError as e:
    print(f"TypeError: {e}")

### 8. Strings de múltiples líneas

En Python, una forma directa de incorporar un salto de línea en una cadena de texto es mediante el uso de la secuencia de escape \n, la cual indica al intérprete el punto en el que debe iniciarse una nueva línea.

In [ ]:
nombre = "Rojo\nAzul\nBlanco"

In [ ]:
print(nombre)

### 9. Números

Python dispone de tres tipos de datos numéricos fundamentales: los enteros (int), utilizados para representar valores sin parte decimal; los números de punto flotante (float), empleados para valores reales con componente decimal; y los números complejos (complex), que incorporan una parte real y una parte imaginaria.

In [ ]:
num = 10

In [ ]:
num2 = int(num)

In [ ]:
print(num2)

En la escritura convencional de números de gran magnitud, es habitual emplear separadores visuales para facilitar su lectura; por ejemplo, el valor 1000000 suele representarse como 1,000,000 o 1.000.000, según la convención numérica utilizada. Sin embargo, en Python no está permitido utilizar comas ni puntos como separadores dentro de un literal numérico, ya que dichos símbolos cumplen otras funciones sintácticas. Para mejorar la legibilidad de números extensos, Python permite emplear el carácter guion bajo (_), de modo que una expresión como 1_000_000 es interpretada correctamente como el número entero un millón.

In [ ]:
num = 123454

In [ ]:
num

In [ ]:
num2 = 1_000_000

In [ ]:
num2

En contraste con los valores enteros, los números de punto flotante permiten representar magnitudes numéricas que incluyen componente decimal. En Python, estos valores se encuentran asociados al tipo de dato float, el cual resulta fundamental para operaciones que requieren precisión numérica continua, como cálculos científicos, análisis de datos y modelamiento computacional.

In [ ]:
num = "20.5"

In [ ]:
num2 = float(num)

In [ ]:
num2

In [ ]:
num3 = 1e6

In [ ]:
num3

Python es uno de los pocos lenguajes de programación de uso general que incorpora soporte nativo para el manejo de números complejos. Aunque este tipo de dato suele asociarse principalmente a contextos de computación científica, modelamiento matemático y procesamiento de señales, también puede resultar relevante en ciertos problemas analíticos y estadísticos que requieren una representación más amplia de magnitudes numéricas.

Un número complejo está compuesto por dos elementos: una parte real y una parte imaginaria. En Python, este tipo de dato se representa mediante la sintaxis a + bj, donde a corresponde a la componente real y b a la componente imaginaria, la cual se expresa utilizando la letra j como unidad imaginaria. Esta convención permite definir y manipular números complejos de forma directa dentro del lenguaje, sin necesidad de bibliotecas adicionales.

In [ ]:
num = 1 + 2j

In [ ]:
num

In [ ]:
num.real

In [ ]:
num.imag

### 10. Comentarios

Los comentarios desempeñan un rol fundamental en el desarrollo de software, al proporcionar un mecanismo de documentación interna que facilita la interpretación de la lógica implementada en el código. Su correcta utilización mejora la trazabilidad, mantenibilidad y legibilidad de los programas, aspectos especialmente relevantes en entornos de análisis de datos, modelamiento algorítmico e inteligencia artificial. Por ello, la incorporación de comentarios claros, precisos y técnicamente pertinentes constituye una práctica recomendada en la construcción de código de calidad.

In [ ]:
var = "Ciencia de Datos" # Esto es una variable

In [ ]:
# Esto es un comentario
# de varias líneas
# en Python

In [ ]:
"""
Esto es un comentario
de varias líneas
en Python
"""

## Ejercicio — sección II

Un registro llega con espacios sobrantes y en minúsculas: `"  pac 42  "`.
Conviértalo en el identificador normalizado `"PAC-00042"`.

Necesitará quitar espacios, separar las dos partes, pasar la primera a mayúsculas
y rellenar el número con ceros hasta cinco dígitos.

In [ ]:
registro = "  pac 42  "

# TODO: transforme 'registro' en "PAC-00042"
resultado = None

print(repr(resultado))
assert resultado == "PAC-00042", "Todavía no coincide con el formato esperado"
print("Correcto")

<details>
<summary><b>Ver solución</b></summary>

```python
prefijo, numero = registro.strip().split()
resultado = f"{prefijo.upper()}-{numero.zfill(5)}"
```

</details>

# III. Funciones

En el ámbito matemático, una función se define como una relación que asocia una o más entradas con una salida determinada, y suele representarse de forma general como y = f(x). No obstante, en programación, el concepto de función adquiere un alcance más amplio y operativo. En este contexto, una función corresponde a una unidad lógica de código que encapsula una tarea específica o un conjunto de operaciones relacionadas, permitiendo su reutilización, organización y abstracción dentro de un programa. Su uso favorece el diseño modular, ya que facilita la descomposición de problemas complejos en componentes más pequeños, comprensibles y mantenibles, aspecto esencial en el desarrollo de algoritmos, sistemas de análisis de datos y aplicaciones de inteligencia artificial.

In [ ]:
var = "Ciencia de Datos"

In [ ]:
len(var)

La función presentada anteriormente forma parte del conjunto de funciones integradas que ofrece el intérprete de Python, las cuales pueden ser utilizadas directamente siempre que resulte necesario. Para hacer uso de estas funciones no es indispensable conocer el detalle de su implementación interna, sino comprender adecuadamente su interfaz, es decir, qué argumentos reciben y qué tipo de valor retornan.

Cuando una función es invocada mediante su nombre y se le proporcionan los argumentos requeridos, el intérprete de Python ejecuta internamente las instrucciones asociadas a su definición, aplicando las operaciones correspondientes sobre los datos de entrada. Una vez finalizada su ejecución, el control del programa retorna al punto exacto desde el cual la función fue llamada. Dependiendo de su diseño, la función puede además entregar un valor de retorno, el cual puede ser almacenado, procesado o reutilizado en etapas posteriores del código.

### 1. Definiendo funciones personalizadas

La sintaxis utilizada para definir funciones en Python es la siguiente:


**Estructura general de una función** (esto es la plantilla, no código ejecutable):

```python
def <nombre_funcion>([<parámetros>]):
    <sentencia(s)>
```

El último elemento, `<sentencia(s)>`, se denomina cuerpo de la función. El cuerpo es el bloque de sentencias en Python que se ejecutará cuando se llame a la función. El cuerpo de una función de Python se define por la sangría.

In [ ]:
# Definir una función NO la ejecuta: solo registra qué hará cuando se llame.
def mi_funcion(arg1, arg2):
    print(arg1)
    print(arg2)

En Python, la llamada a una función se realiza mediante la sintaxis general `<nombre_funcion>([<argumentos>])`. En esta expresión, los argumentos representan los datos concretos que se entregan en el momento de la invocación, mientras que los parámetros corresponden a las variables formales definidas en la estructura de la función para recibir dichos valores. Esta correspondencia entre argumentos y parámetros permite ejecutar procedimientos de manera flexible, modular y reutilizable dentro del programa.

In [ ]:
mi_funcion("Hola mundo", "Adios mundo")

In [ ]:
# El orden de salida muestra el flujo: al llamar la función, la ejecución
# salta a su cuerpo y vuelve al punto siguiente cuando termina.
print(1)
mi_funcion(2, 3)
print(4)

Python permite la definición de funciones sin parámetros y su correspondiente invocación sin argumentos; no obstante, en ambos casos el uso de paréntesis es obligatorio. Esto se debe a que los paréntesis constituyen un elemento sintáctico fundamental en la declaración y llamada de funciones, aun cuando no exista información que transferir entre la función y el contexto desde el cual se ejecuta.

In [ ]:
def mi_funcion2():
    print("Ciencia de Datos")

In [ ]:
mi_funcion2()

### 2. Argumentos de las funciones

En Python, los argumentos que se proporcionan al invocar una función no están restringidos a un único tipo de dato, sino que pueden pertenecer a diversas categorías, incluyendo tipos primitivos y estructuras más complejas. Esta característica aporta versatilidad al diseño funcional del lenguaje y facilita la construcción de soluciones modulares aplicables a distintos contextos de programación, análisis de datos e inteligencia artificial.

### 2.1 Argumentos posicionales

En Python, una de las formas más básicas de transferir información a una función es a través de argumentos posicionales. En este mecanismo, la correspondencia entre argumentos y parámetros se establece según la posición que ocupan en la llamada y en la definición de la función, respectivamente. Por esta razón, la función debe declarar explícitamente, entre paréntesis y separados por comas, los parámetros que definirán la estructura esperada de entrada.

In [ ]:
def mi_funcion(arg1, arg2, arg3):
    print(arg1)
    print(arg2)
    print(arg3)

El número de argumentos que le proporcionamos en la invocación de la función debe respetar el orden y el número de parámetros definidos.

In [ ]:
mi_funcion("Magister", "Ciencia de Datos", "Inteligencia Artificial")

In [ ]:
# Demostración de error: falta un argumento posicional obligatorio.
# Se ejecuta dentro de try/except para que el cuaderno pueda correr
# completo; el mensaje que imprime es exactamente el que verían.
try:
    mi_funcion("Magister", "Ciencia de Datos")
except TypeError as e:
    print(f"TypeError: {e}")

En Python, los parámetros constituyen variables locales asociadas exclusivamente al entorno interno de la función, por lo que su alcance queda restringido al bloque de código que conforma su definición. Durante la invocación, los argumentos suministrados son enlazados a los parámetros conforme al orden en que estos fueron declarados, estableciendo la correspondencia necesaria para la ejecución del procedimiento definido.

In [ ]:
# 'var' existe solo DENTRO de la función: es una variable local. Fuera de
# ella no está definida, y por eso el intento de imprimirla falla.
def mi_funcion():
    var = "variable 'var' dentro de la funcion"
    print(var)

In [ ]:
var = "variable 'var' fuera de la funcion"
mi_funcion()

# Demostración de error: la variable no está definida en este ámbito.
# Se ejecuta dentro de try/except para que el cuaderno pueda correr
# completo; el mensaje que imprime es exactamente el que verían.
try:
    print(var)
except NameError as e:
    print(f"NameError: {e}")

### 2.2 Argumentos de palabra clave


En Python, una función puede invocarse mediante argumentos nombrados, utilizando la estructura `<palabra_clave>=<valor>`. Bajo este mecanismo, cada palabra clave debe coincidir con un parámetro formal declarado en la definición de la función, estableciendo una correspondencia explícita entre los valores suministrados y las variables receptoras. Esta modalidad mejora la legibilidad del código y aporta mayor flexibilidad en la llamada a funciones con múltiples parámetros.



In [ ]:
# Definir una función NO la ejecuta: solo registra qué hará cuando se llame.
def mi_funcion(arg1, arg2):
    print(arg1)
    print(arg2)

In [ ]:
var = "Ciencia de Datos"
var2 = "Inteligencia Artificial"

mi_funcion(arg1=var, arg2=var2)

In [ ]:
# Demostración de error: el nombre del argumento no existe en la definición.
# Se ejecuta dentro de try/except para que el cuaderno pueda correr
# completo; el mensaje que imprime es exactamente el que verían.
try:
    mi_funcion(arg1=var, arg3=var2)
except TypeError as e:
    print(f"TypeError: {e}")

La utilización de argumentos nombrados en Python elimina la dependencia del orden posicional en la llamada a una función, puesto que la asignación entre argumentos y parámetros se establece de manera explícita a través del nombre de cada parámetro.

In [ ]:
mi_funcion(arg1=var, arg2=var2)

In [ ]:
mi_funcion(arg2=var2, arg1=var)

Si bien el uso de argumentos nombrados otorga flexibilidad respecto del orden de asignación, no elimina la necesidad de satisfacer la aridad de la función; es decir, debe proporcionarse el número de argumentos exigido por su definición, excepto en aquellos casos en que existan parámetros opcionales o estructuras variables de entrada.

In [ ]:
mi_funcion(var, arg2=var2)

### 2.3 Parámetros con valores por defecto u opcionales

En Python, cuando un parámetro se define utilizando la sintaxis `<nombre>=<valor>`, el `<valor>` asignado queda establecido como valor por defecto dentro de la firma de la función. Esto implica que su provisión en la llamada deja de ser obligatoria, ya que, en ausencia de un argumento explícito, el intérprete empleará automáticamente el valor predeterminado. Este tipo de parámetros se denomina parámetros opcionales o parámetros con valor por defecto.

In [ ]:
# Un parámetro con valor por defecto puede omitirse al llamar la función.
# Los que tienen valor por defecto van SIEMPRE después de los que no.
def mi_funcion(arg1, arg2="Inteligencia Artificial"):
    print(arg1)
    print(arg2)

In [ ]:
mi_funcion("Ciencia de Datos")

In [ ]:
mi_funcion("Inteligencia Artificial", "Ciberseguridad")

### 3. Sentencia Return

En Python, una función puede devolver un valor una vez finalizada la ejecución de las instrucciones que conforman su cuerpo. Para ello, se utiliza la palabra clave `return`, cuya finalidad es transferir un resultado desde el ámbito interno de la función hacia el contexto desde el cual esta fue invocada. De este modo, el valor retornado puede ser almacenado, reutilizado en otras expresiones o incorporado en etapas posteriores del flujo computacional.

Al utilizar la instrucción `return`, es importante considerar varios aspectos relacionados con el comportamiento de la función y con la forma en que dicho valor de salida será gestionado dentro del programa.

3.1 La ejecución de la sentencia `return` provoca la terminación inmediata del flujo interno de la función, transfiriendo el control nuevamente a la línea de código donde se realizó la llamada, junto con el valor de retorno correspondiente, si este ha sido especificado.

In [ ]:
# return corta la ejecución de la función de inmediato: la línea siguiente
# nunca se alcanza, aunque esté escrita.
def mi_funcion():
    print("Sentencia 1 en mi_funcion")
    return
    print("Sentencia 2 en mi_funcion")

In [ ]:
print("Ciencia de Datos")
mi_funcion()
print("Ciberseguridad")

3.2 Cuando la sentencia `return` adopta la estructura `return` `<valor>`, el intérprete finaliza la ejecución de la función y transfiere el `<valor>` especificado como resultado de salida hacia la llamada que originó su ejecución.

In [ ]:
def mi_funcion():
    return "Valor de mi_funcion"

In [ ]:
var = mi_funcion()

# Demostración de error: la variable no está definida en este ámbito.
# Se ejecuta dentro de try/except para que el cuaderno pueda correr
# completo; el mensaje que imprime es exactamente el que verían.
try:
    print(var)
except NameError as e:
    print(f"NameError: {e}")

In [ ]:
mi_funcion()

In [ ]:
mi_funcion()[0:5]

3.3 Cuando la instrucción `return` no incluye un `<valor>` asociado, Python retorna por defecto `None`, un objeto especial que indica que la función ha finalizado sin producir un resultado explícito.

In [ ]:
def mi_funcion():
    return

In [ ]:
var = mi_funcion()

In [ ]:
# Demostración de error: la variable no está definida en este ámbito.
# Se ejecuta dentro de try/except para que el cuaderno pueda correr
# completo; el mensaje que imprime es exactamente el que verían.
try:
    print(var)
except NameError as e:
    print(f"NameError: {e}")

3.4 Python permite que una función retorne múltiples valores mediante una única sentencia `return`, escribiendo los elementos separados por comas. Aunque sintácticamente se perciben como valores independientes, el intérprete los encapsula internamente en una tupla, lo que facilita su asignación simultánea a múltiples variables o su tratamiento como una estructura compuesta de salida. Devolviendo varios valores utilizando la sintaxis `return <valor1>,<valor2>,...,<valorn>`

In [ ]:
# Se pueden devolver varios valores separados por comas: Python los empaqueta
# en una tupla, que después puede desempaquetarse en tres variables.
def mi_funcion():
    return "texto1","texto2","texto3"

In [ ]:
var1,var2,var3 = mi_funcion()

In [ ]:
print(var1)
print(var2)

# Demostración de error: la variable nunca fue definida.
# Se ejecuta dentro de try/except para que el cuaderno pueda correr
# completo; el mensaje que imprime es exactamente el que verían.
try:
    print(var3)
except NameError as e:
    print(f"NameError: {e}")

### 4. Docstrings

Cuando la primera instrucción dentro del cuerpo de una función en Python corresponde a una cadena de texto delimitada, por ejemplo, mediante comillas triples """texto""", esta se interpreta como el docstring de la función.

El docstring cumple una función documental, ya que permite describir formalmente el propósito de la función, los parámetros que recibe, los valores que retorna y cualquier otra información relevante para su correcta comprensión y uso. Su incorporación favorece la legibilidad, mantenibilidad y reutilización del código, especialmente en proyectos de mayor complejidad o desarrollados de forma colaborativa.

Existe un PEP (Python Enhancement Proporsal) que establece el estilo con el que deben definirse los docstrings: https://www.python.org/dev/peps/pep-0257/

In [ ]:
def mi_funcion():
    """Esto es un docstring de la función."""
    print("Ciencia de Datos")

In [ ]:
mi_funcion()

In [ ]:
def mi_funcion2(arg1=0.0, arg2=0.0):
    """Esto es un docstring de varias líneas.

    Keyword arguments:
    arg1 -- primer argumento de la funcion (default 0.0)
    arg2 -- segundo argumento de la funcion (default 0.0)
    """
    return

In [ ]:
mi_funcion2()

## Ejercicio — sección III

Escriba `resumir(valores, decimales=2)` que reciba una lista de números y
devuelva un diccionario con `n`, `promedio`, `minimo` y `maximo`, redondeando los
tres últimos a los decimales indicados.

La función debe **lanzar `ValueError`** si la lista viene vacía: devolver `None`
en ese caso obliga a quien la use a recordar comprobarlo, y tarde o temprano se
olvida.

In [ ]:
def resumir(valores, decimales=2):
    """TODO: devolver n, promedio, minimo y maximo."""
    pass


mediciones = [110.5, 98.2, 187.4, 142.0, 95.8, 210.3, 130.1]
res = resumir(mediciones)
print(res)

assert res["n"] == 7, "n debe ser la cantidad de valores"
assert res["promedio"] == 139.19, "Revise el redondeo del promedio"
assert res["minimo"] == 95.8 and res["maximo"] == 210.3

try:
    resumir([])
    raise AssertionError("Con la lista vacía debía lanzarse ValueError")
except ValueError:
    print("Correcto")

<details>
<summary><b>Ver solución</b></summary>

```python
def resumir(valores, decimales=2):
    """Resume una lista de números.

    Lanza ValueError si la lista está vacía.
    """
    if not valores:
        raise ValueError("La lista no puede estar vacía")
    return {
        "n": len(valores),
        "promedio": round(sum(valores) / len(valores), decimales),
        "minimo": round(min(valores), decimales),
        "maximo": round(max(valores), decimales),
    }
```

</details>

# IV. Operadores

### 1. Operadores Aritméticos

<table>
<thead>
<tr>
<th>Operator</th>
<th>Example</th>
<th>Meaning</th>
</tr>
</thead>
<tbody>
<tr>
<td><code>+</code>&nbsp;(unario)</td>
<td><code>+a</code></td>
<td><strong>Unario Positivo</strong></td>
</tr>
<tr>
<td><code>+</code>&nbsp;(binario)</td>
<td><code>a + b</code></td>
<td><strong>Suma</strong></td>
</tr>
<tr>
<td><code>-</code>&nbsp;(unario)</td>
<td><code>-a</code></td>
<td><strong>Unario Negativo</strong></td>
</tr>
<tr>
<td><code>-</code>&nbsp;(binario)</td>
<td><code>a - b</code></td>
<td><strong>Resta</strong></td>
</tr>
<tr>
<td><code>*</code></td>
<td><code>a * b</code></td>
<td><strong>Multiplicación</strong></td>
</tr>
<tr>
<td><code>/</code></td>
<td><code>a / b</code></td>
<td><strong>División</strong></td>
</tr>
<tr>
<td><code>%</code></td>
<td><code>a % b</code></td>
<td><strong>Módulo</strong></td>
</tr>
<tr>
<td><code>//</code></td>
<td><code>a // b</code></td>
<td><strong>División de enteros</strong> (también denominado <strong>Floor Division</strong>)</td>
</tr>
<tr>
<td><code>**</code></td>
<td><code>a ** b</code></td>
<td><strong>Exponencial</strong></td>
</tr>
</tbody>
</table>

### Ejemplos con operadores

In [ ]:
num1 = 20

In [ ]:
+num1

In [ ]:
num2 = 5

In [ ]:
num1 + num2

In [ ]:
num1 - num2

In [ ]:
num3 = 2.5
num4 = 1.5

In [ ]:
num3 + num4

In [ ]:
num1 * num2

In [ ]:
num1 / num2 #Siempre el resultado en un float

In [ ]:
text1 = "Hola"
text2 = "mundo"

In [ ]:
num1 % num2 #El operador **Módulo** es un operador binario que devuelve el resto de una división entre tipos de datos numéricos.

In [ ]:
num1 ** 2 #El operador **Exponencial** es un operador binario que se aplica sobre tipos de datos numéricos.

In [ ]:
num1 ** 3

In [ ]:
num1 / num2

In [ ]:
num1 // num2 #El operador Floor Division es un operador binario que se aplica sobre tipos de datos numéricos y devuelve la parte entera del resultado.

In [ ]:
num1 % num2 #Módulo

### 2. Operadores de asignación

Python incorpora una construcción sintáctica que integra, en una sola expresión, una operación aritmética y una asignación. Este mecanismo, denominado augmented assignment, permite modificar el valor de una variable de forma más concisa, reduciendo redundancia y mejorando la claridad del código.

<table>
<thead>
<tr>
<th>Augmented<br>Assignment</th>
<th></th>
<th>Standard<br>Assignment</th>
</tr>
</thead>
<tbody>
<tr>
<td><code>a += 5</code></td>
<td>es equivalente a</td>
<td><code>a = a + 5</code></td>
</tr>
<tr>
<td><code>a /= 10</code></td>
<td>es equivalente a</td>
<td><code>a = a / 10</code></td>
</tr>
<tr>
<td><code>a *= b</code></td>
<td>es equivalente a</td>
<td><code>a = a * b</code></td>
</tr>
</tbody>
</table>

In [ ]:
num = 10

In [ ]:
num *= 2

In [ ]:
num

### 3. Booleanos

En Python, el tipo de dato bool se utiliza para modelar resultados lógicos binarios asociados a la evaluación de expresiones condicionales. Este tipo admite únicamente dos valores posibles: `True` y `False`, los cuales representan, respectivamente, la veracidad o falsedad de una condición. Los booleanos son esenciales en la definición de estructuras de control, la toma de decisiones algorítmicas y la implementación de procesos computacionales basados en lógica formal.

### 4. Operadores de comparación

En Python, los operadores de comparación constituyen mecanismos lógicos que permiten contrastar dos valores y producir un resultado de tipo `bool`, es decir, `True` o `False`. Estos operadores son fundamentales para la construcción de expresiones condicionales, ya que posibilitan evaluar distintas relaciones de equivalencia, desigualdad y orden entre datos. Existen múltiples operadores de comparación, cada uno diseñado para representar una condición relacional específica dentro del flujo del programa.

<table>
<thead>
<tr>
<th>Operador</th>
<th>Ejemplo</th>
<th>Significado</th>
</tr>
</thead>
<tbody>
<tr>
<td><code>==</code></td>
<td><code>a == b</code></td>
<td><strong>Igual a</strong></td>
</tr>
<tr>
<td><code>!=</code></td>
<td><code>a != b</code></td>
<td><strong>No igual a</strong></td>
</tr>
<tr>
<td><code>&lt;</code></td>
<td><code>a &lt; b</code></td>
<td><strong>Menor que</strong></td>
</tr>
<tr>
<td><code>&lt;=</code></td>
<td><code>a &lt;= b</code></td>
<td><strong>Menor que o igual a</strong></td>
</tr>
<tr>
<td><code>&gt;</code></td>
<td><code>a &gt; b</code></td>
<td><strong>Mayor que</strong></td>
</tr>
<tr>
<td><code>&gt;=</code></td>
<td><code>a &gt;= b</code></td>
<td><strong>Mayor que o igual a</strong></td>
</tr>
</tbody>
</table>

In [ ]:
text1 = "cadena de texto"
text2 = "cadena de texto 2"

In [ ]:
text1 == text2

In [ ]:
num1 = 2
num2 = 5

In [ ]:
num1 < num2

### 4. Operadores de lógicos

En Python, los operadores lógicos actúan sobre expresiones booleanas para negar su valor, o bien para combinarlas según relaciones de conjunción y disyunción. Esto permite modelar condiciones más elaboradas y representar reglas de decisión complejas dentro del flujo de ejecución de un programa.

In [ ]:
num = 5

In [ ]:
num < 10

In [ ]:
not num < 10. #Operador not

In [ ]:
num1 = 52
num2 = 100

In [ ]:
num1 < 4 or num2 > 5 #Operador or

In [ ]:
num1 < 4 and num2 > 5 #Operador and

## Ejercicio — sección IV

Complete `hay_medicion(valor)` de modo que distinga correctamente **un cero
legítimo** de la ausencia de dato.

Debe devolver `True` para cualquier número —incluido el `0`— y `False` solo
cuando el valor sea `None`. Es el error que la sección advirtió: `if not valor`
trata el cero como si faltara.

In [ ]:
def hay_medicion(valor):
    """TODO: True si hay un valor medido; False solo si es None."""
    pass


assert hay_medicion(110.5) is True
assert hay_medicion(0) is True, "Un 0 medido SÍ es un dato"
assert hay_medicion(0.0) is True
assert hay_medicion(None) is False
print("Correcto: el 0 se distingue de la ausencia de dato")

<details>
<summary><b>Ver solución</b></summary>

```python
def hay_medicion(valor):
    return valor is not None
```

</details>

# V. Tipos de Datos Complejos

### 1. Listas

En Python, las listas corresponden a estructuras de datos compuestas que permiten representar secuencias ordenadas de elementos heterogéneos. Su principal fortaleza radica en su flexibilidad, ya que admiten inserciones, eliminaciones y modificaciones dinámicas, a diferencia de los arreglos tradicionales de otros lenguajes, que suelen imponer mayores restricciones en cuanto a tipo y tamaño.

El tipo de dato `list` en Python corresponde a una estructura secuencial mutable, cuya sintaxis de definición consiste en encerrar entre corchetes una colección de elementos separados por comas, conforme al esquema `[obj1, obj2, ..., objn]`. Esta notación permite construir de manera explícita una secuencia ordenada de objetos, potencialmente heterogéneos, sobre la cual pueden aplicarse múltiples operaciones de acceso, actualización y transformación.

In [ ]:
lista = [1, 2, 3, 4, 5]

In [ ]:
print(lista)

In [ ]:
lista2 = ["texto1", "texto2", "texto3"]

In [ ]:
type(lista2)

In [ ]:
print(lista2)

Una característica fundamental de las listas en Python es su naturaleza ordenada. Esto implica que cada elemento ocupa una posición específica dentro de la secuencia, y que dicha disposición se mantiene de manera consistente durante el ciclo de vida de la lista, a menos que una operación de inserción, eliminación o reordenamiento altere explícitamente su estructura.

In [ ]:
lista1 = ['t1', 't2', 't3']

In [ ]:
print(lista1)

In [ ]:
lista2 = ['t2', 't1', 't3']

In [ ]:
lista1 == lista2

In [ ]:
lista1 == ['t1', 't2', 't3']

Una de las propiedades distintivas de las listas en Python es su capacidad para contener elementos heterogéneos, es decir, objetos pertenecientes a diferentes tipos de datos dentro de una misma estructura secuencial. Esta característica amplía su utilidad en escenarios donde se requiere modelar información diversa de manera unificada.

In [ ]:
lista = [1, 2, "uno", "dos"]

In [ ]:
print(lista)

Dado que en Python las funciones son tratadas como objetos de primera clase, es posible incluirlas como elementos dentro de una lista. Esta característica amplía notablemente la flexibilidad del lenguaje, ya que permite almacenar, pasar y manipular funciones de manera dinámica dentro de estructuras de datos. Lo veremos mas adelante.

In [ ]:
def func():
    print("Ciencia de Datos")

In [ ]:
lista = ["texto1", "texto2", func]

In [ ]:
print(lista)

En Python, las listas permiten la presencia de elementos duplicados, ya que no imponen unicidad sobre los objetos almacenados. En consecuencia, un mismo elemento puede incorporarse varias veces y conservar cada una de sus ocurrencias según la posición que ocupe dentro de la secuencia.

In [ ]:
lista = ["texto", "texto", "texto"]

In [ ]:
print(lista)

Los elementos de una lista en Python pueden recuperarse mediante el mismo mecanismo de indexación utilizado en las cadenas de texto. Dado que las listas también constituyen estructuras secuenciales ordenadas, cada elemento ocupa una posición específica dentro de la colección y puede ser accedido a través de su índice correspondiente.

Asimismo, todas las variantes de acceso vistas previamente para strings, tales como la indexación simple (indexing), la extracción de subsecuencias (slicing) y el uso de pasos de recorrido (stride), son igualmente aplicables a las listas. Esto permite no solo acceder a elementos individuales, sino también seleccionar subconjuntos de elementos o recorrer la secuencia con intervalos definidos.

### Indexing

In [ ]:
lista = ["text1", "text2", "text3", "text4", "text5"]

In [ ]:
lista[2]

In [ ]:
lista[-1]

### Slicing

In [ ]:
lista[2:4]

In [ ]:
lista[:3]

In [ ]:
lista[2:]

### Stride

In [ ]:
lista[0:4:2]

El mecanismo de stride no solo permite controlar el intervalo de recorrido sobre una lista, sino también modificar la dirección en que esta es recorrida. En particular, al emplear un paso negativo en una operación de slicing, es posible obtener una versión invertida de la lista. Por ejemplo, la expresión lista[::-1] recorre la secuencia desde el último elemento hasta el primero, produciendo como resultado una nueva lista en orden inverso.

In [ ]:
lista

In [ ]:
lista[::-1]

Aunque los mecanismos de indexación y slicing son en gran medida análogos entre cadenas de texto y listas en Python, una de las pocas construcciones cuya interpretación práctica presenta una diferencia relevante es la expresión `[:]`.

a.  Al utilizar la expresión [:] sobre un string, Python devuelve una referencia equivalente a la cadena original, sin necesidad de crear una nueva estructura modificable, debido al carácter inmutable de este tipo de dato.

In [ ]:
texto = "Ciencia de Datos"

In [ ]:
texto[:]

b. A diferencia de lo que ocurre con las cadenas, al utilizar la expresión [:] sobre una lista, Python genera una nueva instancia que contiene los mismos elementos que la lista original. En consecuencia, se obtiene una copia superficial de la estructura, y no una referencia al mismo objeto en memoria.

In [ ]:
lista = [1, 2, 3, 4]

In [ ]:
lista[:]

Como se indicó previamente, las listas en Python admiten la aplicación de diversos operadores y funciones predefinidas del lenguaje, lo que permite efectuar sobre ellas múltiples operaciones de acceso, modificación, evaluación y procesamiento de manera directa y flexible.

In [ ]:
# Con listas, el + CONCATENA en vez de sumar elemento a elemento. Para lo
# segundo hacen falta los arreglos de NumPy, que se ven más adelante.
lista1 = [1, 2, 3]
lista2 = [4, 5, 6]

In [ ]:
lista1 + lista2

In [ ]:
lista1 * 2

In [ ]:
len(lista1)

In [ ]:
min(lista1)

In [ ]:
max(lista2)

In [ ]:
[1, 2, 3] + [4]

Dado que las listas en Python pueden contener objetos de naturaleza heterogénea, también es posible que incluyan otras listas como elementos. Esto da lugar a estructuras jerárquicas o anidadas, en las cuales una lista puede contener sublistas, y estas, a su vez, nuevas sublistas, sin una restricción fija en su nivel de profundidad. Este tipo de organización resulta especialmente útil para modelar datos multidimensionales o representaciones estructuradas de mayor complejidad.

In [ ]:
lista = [1, [2, [3, 4], 5], 6]

In [ ]:
lista[1]

El acceso a elementos ubicados en listas anidadas se realiza mediante indexación sucesiva. En Python, esto se expresa con la sintaxis [][], en la que cada par de corchetes representa un nivel de acceso dentro de la estructura jerárquica de la lista. De este modo, el primer índice identifica la sublista y el segundo permite recuperar el elemento contenido en ella.

In [ ]:
lista[1][0]

In [ ]:
lista[1][1]

In [ ]:
lista[1][1][0]

En Python, las listas pueden anidarse con una profundidad arbitraria, limitada esencialmente por los recursos de memoria y las capacidades de ejecución del entorno. Sin embargo, las operaciones aplicadas sobre una lista anidada afectan, por defecto, solo a la estructura del nivel superior, sin extender su efecto de manera recursiva a los niveles internos que la componen.

In [ ]:
lista

En contraste con los strings, que son inmutables, las listas en Python poseen mutabilidad estructural, lo que permite alterar su contenido una vez instanciadas. Esta propiedad posibilita la modificación directa de sus elementos, así como la inserción, eliminación y reorganización de objetos dentro de la misma estructura de datos.

In [ ]:
lista = ["texto1", "texto2", "texto3"]

In [ ]:
lista[0]

In [ ]:
lista[0] = "texto4"

In [ ]:
lista

También podemos eliminar elementos de una lista.

In [ ]:
del lista[0]

In [ ]:
lista

Podemos utilizar otros conceptos de indexación, como el slicing o el stride para modificar varios elementos de una lista

In [ ]:
lista

In [ ]:
lista += ["texto4", "texto5", "texto6"]

In [ ]:
lista

In [ ]:
lista[0:3]

In [ ]:
lista[0:3] = [1, 2, 3]

In [ ]:
lista

El número de elementos seleccionados no tiene que ser igual a los asignados.

In [ ]:
lista[0:3]

In [ ]:
lista[0:3] = [1, 2]

In [ ]:
lista

In [ ]:
lista[2:2] = [3, 4, 5]

In [ ]:
lista

In [ ]:
lista[0:6]

In [ ]:
lista[0:6] = []

In [ ]:
lista

## 2. Tuplas

En Python, una tupla es una estructura secuencial ordenada que comparte varias propiedades con las listas, pero difiere de ellas principalmente en su sintaxis y en su comportamiento frente a la modificación. Las tuplas se definen mediante paréntesis, siguiendo la forma `(obj1, obj2, ..., objn)`, y se caracterizan por ser inmutables, es decir, no admiten alteraciones en sus elementos después de su instanciación. Esta propiedad las hace especialmente útiles en contextos donde se requiere preservar la integridad de los datos.

In [ ]:
tupla = (1, 2, 3, 4, 5)

In [ ]:
print(tupla)

In [ ]:
type(tupla)

Estructura y funcionamiento de las tuplas en Python

In [ ]:
tupla = (1, 2, 3, "text1", "text2", "text3")

In [ ]:
tupla

In [ ]:
tupla[0:3]

In [ ]:
# Demostración de error: las tuplas son inmutables.
# Se ejecuta dentro de try/except para que el cuaderno pueda correr
# completo; el mensaje que imprime es exactamente el que verían.
try:
    tupla[2] = "text0"
except TypeError as e:
    print(f"TypeError: {e}")

¿Cuándo utilizar una tupla en lugar de una lista?
Desde una perspectiva de diseño y eficiencia computacional, las tuplas resultan preferibles a las listas en contextos donde la colección de datos no requiere modificaciones posteriores a su creación. En primer lugar, las tuplas suelen presentar un rendimiento ligeramente superior en términos de almacenamiento y acceso, debido a su naturaleza inmutable y a una implementación más liviana que la de las listas, aunque esta diferencia adquiere mayor relevancia en estructuras de datos de tamaño considerable. En segundo lugar, la inmutabilidad de las tuplas proporciona una garantía adicional de integridad, ya que evita alteraciones accidentales sobre los valores almacenados, aspecto especialmente importante en el desarrollo de programas robustos y mantenibles. Finalmente, ciertos componentes del lenguaje, como las claves de los diccionarios, exigen objetos inmutables y hashables; en este escenario, las tuplas pueden utilizarse válidamente, siempre que sus elementos también sean inmutables, mientras que las listas quedan excluidas debido a su carácter mutable.

Se debe tener especial cuidado al definir una tupla de un solo elemento cuyo contenido sea numérico, ya que Python puede interpretar los paréntesis únicamente como un mecanismo de agrupación de una expresión aritmética, y no como la construcción de una tupla. En consecuencia, para que el intérprete reconozca correctamente una tupla unitaria, es necesario incluir una coma después del único elemento, por ejemplo: (5,). De lo contrario, una expresión como (5) será tratada simplemente como un valor numérico y no como una tupla.

In [ ]:
tupla = (2)

In [ ]:
tupla

In [ ]:
type(tupla)

La forma correcta de resolver esta ambigüedad sintáctica consiste en definir la tupla mediante la expresión (num,), ya que la presencia de la coma es el elemento que permite a Python reconocer la estructura como una tupla unitaria. Sin dicha coma, el valor sería interpretado únicamente como un objeto numérico agrupado entre paréntesis.

In [ ]:
tupla = (2,)

In [ ]:
tupla

In [ ]:
type(tupla)

Empaquetamiento (packing) y desempaquetamiento (unpacking) de tuplas
En las secciones anteriores se observó que una tupla compuesta por múltiples elementos puede ser asignada íntegramente a una única variable. Este proceso se conoce como packing, ya que consiste en agrupar varios valores dentro de una sola estructura de datos.

In [ ]:
tupla = (1, 2, 3, 4)

Sin embargo, las tuplas no solo permiten agrupar múltiples valores en una única estructura, sino también descomponer dicha estructura y asignar cada uno de sus elementos a variables distintas en una sola operación. Este mecanismo, denominado unpacking, facilita una manipulación más clara, concisa y expresiva de los datos.

In [ ]:
num1, num2, num3, num4 = tupla

In [ ]:
num1

In [ ]:
num4

In [ ]:
# Demostración: el número de variables debe coincidir con el de elementos.
# Se captura el ValueError para que el cuaderno siga corriendo.
try:
    num1, num2, num3 = tupla
except ValueError as e:
    print(f"ValueError: {e}")
    print(f"  la tupla tiene {len(tupla)} elementos y se pidieron 3")

El proceso de unpacking corresponde al mismo principio empleado previamente al trabajar con funciones que retornan múltiples valores, ya que permite descomponer la tupla generada implícitamente por return y asignar cada uno de sus elementos a variables distintas en una sola operación.

In [ ]:
def func():
    return 5,6

In [ ]:
func()

In [ ]:
num1, num2 = func()

In [ ]:
num1

In [ ]:
num2

Una característica sintáctica relevante de las tuplas en Python es que su definición no depende obligatoriamente del uso de paréntesis, sino de la presencia de valores separados por comas. En consecuencia, al escribir una sentencia como return valor1, valor2, valor3, el intérprete construye implícitamente una tupla con esos elementos y la utiliza como valor de retorno de la función. Los paréntesis, por tanto, cumplen principalmente una función de claridad o desambiguación sintáctica, pero no son un requisito indispensable en este contexto.

In [ ]:
tupla = 1, 3, 5

In [ ]:
tupla

## 3. Diccionario

En Python, los diccionarios constituyen una estructura de datos mutable basada en un esquema de mapeo entre claves y valores. Cada entrada del diccionario establece una correspondencia entre una clave única y un valor asociado, permitiendo organizar y recuperar información de manera eficiente a partir de identificadores explícitos. Esta estructura resulta fundamental en múltiples contextos de programación, análisis de datos e inteligencia artificial, especialmente cuando se requiere representar atributos, configuraciones, registros o relaciones no secuenciales entre entidades.

El tipo de dato dict en Python corresponde a una estructura de mapeo mutable, cuya sintaxis de definición consiste en una colección de pares clave-valor delimitados por llaves y separados por comas, conforme al esquema `{clave1: valor1, clave2: valor2, ..., claven: valorn}`. En esta estructura, cada clave funciona como un identificador único que permite acceder al valor asociado de manera directa, lo que convierte a los diccionarios en una herramienta fundamental para modelar relaciones semánticas y representar información estructurada.

In [ ]:
dic = {
    "Nombre":"Luis",
    "Apellido":"Silva",
    "Pais":"Brasil",
    "Ciudad":"Rio de Janeiro"
}

In [ ]:
type(dic)

In [ ]:
print(dic)

In [ ]:
# Otra forma de definir diccionarios con la funcion dict()
dic2 = dict(
    Nombre="Luis",
    Apellido="Silva",
    Pais="Brasil",
    Ciudad="Rio de Janeiro"
)

In [ ]:
print(dic2)

A diferencia de las estructuras secuenciales como listas y tuplas, los diccionarios en Python no admiten acceso posicional mediante índices numéricos. Su mecanismo de recuperación de información se basa en claves, de modo que cada valor se obtiene especificando la clave correspondiente entre corchetes `[]`. Esta forma de acceso refleja la naturaleza no secuencial del diccionario y su lógica de mapeo entre identificadores y valores asociados.

In [ ]:
dic = {
    "Nombre":"Luis",
    "Apellido":"Silva",
    "Pais":"Brasil",
    "Ciudad":"Rio de Janeiro"
}

In [ ]:
dic['Nombre']

In [ ]:
dic['Pais']

En Python, las claves de un diccionario deben ser objetos inmutables y hashables, ya que el mecanismo interno de almacenamiento y acceso de esta estructura depende de la estabilidad de su valor. Por esta razón, es posible utilizar como claves elementos tales como enteros, strings y tuplas, mientras que estructuras mutables, como listas o diccionarios, no son válidas para este propósito.

In [ ]:
# Las CLAVES pueden ser de cualquier tipo inmutable, no solo texto:
# aquí son enteros. Lo que no puede ser clave es una lista.
dic2 = {
    0: "cero",
    1: "uno",
    2: "dos",
    3: "tres"
}

In [ ]:
dic2[0]

In [ ]:
dic2[3]

In [ ]:
# Una tupla sí puede ser clave, porque es inmutable. Esto permite indexar
# por combinaciones, por ejemplo (comuna, año).
dic3 = {
    ("uno", 1): "one",
    ("dos", 2): "two",
    ("tres", 3): "three"
}

In [ ]:
dic3[("uno", 1)]

In [ ]:
dic3["dos", 2]

Los diccionarios en Python son estructuras de datos mutables basadas en asociaciones clave-valor. A diferencia de las listas y las tuplas, sus elementos no se acceden mediante índices numéricos, sino a través de claves, lo que responde a su naturaleza de mapeo y no de secuencia. No obstante, en las versiones actuales de Python, los diccionarios sí preservan el orden de inserción de sus elementos, aunque su mecanismo de acceso continúa estando determinado por la clave y no por la posición.

In [ ]:
dic = {
    "Nombre":"Luis",
    "Apellido":"Silva",
    "Pais":"Brasil",
    "Ciudad":"Rio de Janeiro"
}

In [ ]:
dic["Nombre"] = "Pedro"

In [ ]:
dic["Apellido"] = "Ramos"

In [ ]:
dic["Edad"] = 30

In [ ]:
print(dic)

Tal como se aprecia en el ejemplo anterior, los diccionarios en Python admiten la asignación de valores de distintos tipos de datos a las claves que los componen. Esta característica permite almacenar desde tipos simples, como números y strings, hasta estructuras más complejas, como listas, tuplas o incluso otros diccionarios, ampliando significativamente su capacidad para modelar relaciones y estructuras de datos heterogéneas.

In [ ]:
# Los VALORES pueden ser de cualquier tipo, incluidas listas, tuplas y
# otros diccionarios: así se representan estructuras anidadas como un JSON.
dic2 = {
    "num":10,
    "str":"Ciencia de Datos",
    "lista":[1, 2, 3, 4],
    "tupla":(1, 2, 3, 4),
    "dic":{"k1":"clave1", "k2":"clave2"}
}

In [ ]:
dic2["str"]

In [ ]:
dic2["lista"]

In [ ]:
dic2["lista"][2]


In [ ]:
dic2["dic"]

In [ ]:
dic2["dic"]["k1"]

Las claves de un diccionario en Python tampoco están obligadas a pertenecer a un único tipo de dato. En efecto, es posible definir claves de distinta naturaleza, siempre que correspondan a objetos inmutables y hashables, condición necesaria para garantizar su uso válido dentro de esta estructura.

In [ ]:
# Un mismo diccionario puede mezclar tipos de clave. Es válido, pero conviene
# evitarlo: dificulta recorrerlo y razonar sobre él.
dic3 = {
    1:"primer valor",
    "dos":"segundo valor",
    ("tres",):"tercer valor"
}

In [ ]:
dic3[1]

In [ ]:
dic3["dos"]

In [ ]:
dic3[("tres",)]

La restricción más relevante que debe considerarse al trabajar con diccionarios en Python es que cada clave debe ser única dentro de la estructura. En consecuencia, no es posible que un diccionario contenga dos entradas con la misma clave, ya que cada una de ellas actúa como un identificador exclusivo del valor que tiene asociado.

In [ ]:
# Las claves NO se repiten: si una se declara dos veces, la última gana
# silenciosamente. Es una fuente clásica de datos perdidos sin aviso.
dic4 = {
    "key":"value1",
    "key":"value2"
}

In [ ]:
dic4

Al igual que otras estructuras de datos complejas en Python, los diccionarios son compatibles con distintos operadores y funciones predefinidas del lenguaje. Esta compatibilidad permite realizar sobre ellos múltiples operaciones de acceso, manipulación, evaluación y procesamiento, facilitando su utilización en contextos donde se requiere representar y gestionar información mediante asociaciones clave-valor.

In [ ]:
dic1 = {
    "key1": "value1",
    "key2": "value2"
}

In [ ]:
dic2 = {
    "key3": "value3",
    "key4": "value4"
}

In [ ]:
dic1 == dic2

In [ ]:
dic1 == dic1

In [ ]:
dic1 is dic1

In [ ]:
dic1 is dic2

La siguiente operación no es posible.

In [ ]:
# Demostración de error: los diccionarios no se concatenan con +.
# Se ejecuta dentro de try/except para que el cuaderno pueda correr
# completo; el mensaje que imprime es exactamente el que verían.
try:
    dic1 + dic2
except TypeError as e:
    print(f"TypeError: {e}")

## 4. Sets

Los conjuntos (sets) en Python son estructuras de datos que permiten almacenar múltiples elementos dentro de una misma variable. A diferencia de las listas, los sets no constituyen colecciones indexadas ni ordenadas, por lo que sus elementos no pueden recuperarse mediante posiciones específicas. Además, esta estructura no admite valores duplicados, ya que cada elemento debe ser único dentro del conjunto. Estas propiedades los convierten en una alternativa especialmente útil para operaciones de pertenencia, eliminación de duplicados y análisis de relaciones entre colecciones de datos.

Los set se representan dentro de python con el tipo de dato `set`. La sintaxis que se utiliza para definir un set en Python es la siguiente: `{val1, val2, ..., valn}`

In [ ]:
miset = {"rojo", "azul", "verde"}

In [ ]:
print(miset)

In [ ]:
type(miset)

Funcionamiento de un conjunto (set)
Debido a las propiedades estructurales de este tipo de dato, los sets suelen emplearse en contextos específicos donde resulta necesario trabajar con colecciones de elementos únicos. Entre sus aplicaciones más frecuentes se encuentran la eliminación de valores duplicados en una secuencia de datos y la implementación de operaciones propias de la teoría de conjuntos, como la unión, la intersección y la diferencia. Estas características convierten a los sets en una estructura especialmente útil para tareas de depuración, comparación y análisis de datos.

In [ ]:
miset = {"rojo", "azul", "verde"}

In [ ]:
# Demostración: un set NO tiene orden, así que no admite indexación.
# Se captura el error para que el cuaderno pueda correr completo.
try:
    miset[0]
except TypeError as e:
    print(f"TypeError: {e}")
    print("  Para recorrer un set se usa un bucle, o se convierte con list(miset).")

In [ ]:
lista = ["rojo", "rojo", "amarillo", "azul", "azul"]

In [ ]:
set(lista)

In [ ]:
lista_unicos = list(set(lista))

In [ ]:
lista_unicos

Es un tipo de dato exactamente igual que un set, con la diferencia de que es inmutable.

In [ ]:
fset = frozenset({"azul, amarillo, verde"})

In [ ]:
fset

In [ ]:
type(fset)

## Ejercicio — sección V

Dada una lista de diagnósticos con repeticiones, construya:

1. `unicos`: un **conjunto** con los diagnósticos distintos.
2. `conteo`: un **diccionario** con cuántas veces aparece cada uno.
3. `copia_segura`: una copia de la lista original que no se altere al modificar
   la lista.

In [ ]:
diagnosticos = ["hipertension", "diabetes", "hipertension",
                "obesidad", "diabetes", "hipertension"]

unicos = None        # TODO
conteo = None        # TODO
copia_segura = None  # TODO

diagnosticos.append("asma")   # se modifica DESPUÉS de copiar

print("Únicos:", unicos)
print("Conteo:", conteo)

assert unicos == {"hipertension", "diabetes", "obesidad"}
assert conteo["hipertension"] == 3 and conteo["diabetes"] == 2
assert len(copia_segura) == 6, "La copia no debía cambiar al modificar el original"
print("Correcto")

<details>
<summary><b>Ver solución</b></summary>

```python
unicos = set(diagnosticos)
conteo = {d: diagnosticos.count(d) for d in unicos}
copia_segura = diagnosticos.copy()      # o diagnosticos[:]
```

</details>

# VI. Estructuras de control del flujo de ejecución en Python

Las estructuras de control del flujo de ejecución en Python constituyen un componente fundamental en la construcción de algoritmos, ya que permiten regular el orden en que se ejecutan las instrucciones de un programa en función de determinadas condiciones o repeticiones. Mediante mecanismos como las sentencias condicionales y los bucles, es posible modelar procesos de decisión, iteración y adaptación dinámica del comportamiento computacional. En este sentido, su dominio resulta esencial para el desarrollo de soluciones robustas y eficientes en ámbitos como la ciencia de datos, la automatización y la inteligencia artificial.

Hasta este punto, los elementos estudiados en Python se han ejecutado de manera secuencial, es decir, siguiendo estrictamente el orden en que fueron escritos. Sin embargo, en el desarrollo de programas reales, con frecuencia es necesario alterar ese comportamiento para omitir ciertas instrucciones, repetir bloques de código o seleccionar entre diferentes cursos de acción según determinadas condiciones. En este contexto, las estructuras de control de flujo constituyen mecanismos fundamentales, ya que permiten modificar de manera explícita la secuencia de ejecución de un programa y dotarlo de mayor flexibilidad, dinamismo y capacidad de decisión.

## 1. Sentencia `if`

La sintaxis:
```
if <expresión>:
    <sentencia(s)>
```
  
`<expresión>` es una expresión evaluada en un contexto booleano. El resultado de evaluar esta expresión debe ser `True` o `False`

`<sentencia(s)>` es el bloque de sentencias en Python que se ejecutará cuando el resultado de evaluar la expresión sea `True`, en caso contrario se omitirá.

In [ ]:
# La INDENTACIÓN define el bloque: las tres líneas sangradas pertenecen al if
# y se ejecutan juntas, o ninguna. En Python la sangría es sintaxis, no estilo.
num1 = 25
num2 = 30

if num1 < num2:
    print("Sentencia 1")
    print("Sentencia 2")
    print("Sentencia 3")

In [ ]:
# Si la condición es falsa, el bloque completo se salta sin ejecutar nada.
if num1 > num2:
    print("Sentencia 1")
    print("Sentencia 2")
    print("Sentencia 3")

Importante: En Python, todo código que pertenece a una estructura local, como una función o una sentencia de control (if, for, while), debe estar correctamente indentado. La sangría no es solo una convención de estilo, sino un elemento sintáctico esencial para definir los bloques de código y su alcance dentro del programa.

In [ ]:
# La última línea NO está sangrada, así que queda fuera del if:
# se ejecuta siempre, se cumpla o no la condición.
if num1 < num2:
    print("Sentencia 1")
    print("Sentencia 2")
    print("Sentencia 3")
print("Sentencia fuera del if")

In [ ]:
# El operador 'in' comprueba pertenencia: funciona con listas, tuplas,
# conjuntos, diccionarios (busca en las claves) y cadenas de texto.
lista = ["azul", "amarillo", "verde"]

if "azul" in lista:
    print("Sentencia 1 en if")
    print("Sentencia 2 en if")

In [ ]:
tupla = (1, 2, 3, 4)

if 6 in tupla:
    print("Sentencia 1 en if")
    print("Sentencia 2 en if")

La sintaxis para definir la cláusula `else` dentro de una estructura `if` es la siguiente:

```
if <expresión>:
    <sentencia(s)>
else:
    <sentencia(s)>
```

In [ ]:
if num1 > num2:
    print("Sentencias si True")
else:
    print("Sentencias si False")

Otro de los escenarios habituales en programación consiste en la implementación de estructuras condicionales que requieren evaluar múltiples expresiones para determinar el flujo de ejecución adecuado. En estos casos, Python proporciona la cláusula elif, la cual permite encadenar condiciones adicionales dentro de una estructura if, evitando el uso de múltiples bloques independientes y mejorando la claridad del código.

La sintaxis general para definir una estructura condicional con múltiples evaluaciones en Python 3 es la siguiente:

```
if <expresión>:
    <sentencia(s)>
elif <expresión>:
    <sentencia(s)>
elif <expresión>:
    <sentencia(s)
...
else:
    <sentencia(s)>

```

In [ ]:
# elif evalúa en ORDEN y se detiene en la primera condición verdadera:
# aunque hubiera varias que se cumplen, solo entra en una.
nombres = ["Santiago", "Laura", "Julia"]
if "Pedro" in nombres:
    print("Hola Pedro")
elif "Juan" in nombres:
    print("Hola Juan")
elif "Marta" in nombres:
    print("Hola Marta")
elif "Santiago" in nombres:
    print("Hola Santiago")
else:
    print("El nombre no esta en la lista")

## 2. Sentencia `for`

La sentencia for constituye una de las estructuras de control de flujo más fundamentales en programación y forma parte del conjunto de construcciones esenciales del lenguaje Python. Su función principal es permitir la ejecución repetida de un bloque de código a partir del recorrido de una secuencia o colección de elementos. Debido a esta capacidad de repetición controlada, este tipo de estructura se clasifica dentro de los denominados bucles.

En Python, la sentencia for se utiliza habitualmente cuando se desea iterar un número finito de veces sobre una secuencia, como una lista, una tupla, un diccionario, un conjunto o un rango de valores. Su sintaxis general es la siguiente:

La sintaxis utilizada para definir la sentencia `for` es la siguiente:
```
for <variable> in <iterable>:
    <sentencia(s)>
```
  
`<iterable>` es una colección de elementos, por ejemplo, una lista o una tupla.

`<variable>` es una variable que recibirá uno de los elementos del objeto `<iterable>` en cada una de las itreaciones.

`<sentencia(s)>` es el bloque de sentencias en Python que se ejecutará repetidamente.

In [ ]:
colores = ["verde", "azul", "rojo"]
for color in colores:
    print(color)

In [ ]:
# El else de un for se ejecuta cuando el bucle recorre TODOS los elementos
# sin encontrar un break. No es un «si no» como el del if.
for i in ["a", "b", "c"]:
    print(i)
else:
    print("El bucle ya no tiene más elementos")

In [ ]:
# range(5) genera 0, 1, 2, 3, 4: empieza en 0 y el extremo NO se incluye.
for num in range(5):
    print(num)

## 3. Sentencia `while`

La sentencia while constituye otra de las estructuras fundamentales de control de flujo en Python. Su propósito es ejecutar de manera repetida un bloque de código mientras una condición lógica determinada permanezca verdadera. A diferencia de la sentencia for, que suele emplearse cuando el número de iteraciones es conocido o finito, while resulta especialmente útil en situaciones donde la repetición depende de una condición dinámica evaluada durante la ejecución del programa.

En este sentido, la sentencia while puede dar lugar a un número indeterminado de iteraciones y, si la condición nunca deja de cumplirse, incluso a un bucle infinito. Su sintaxis general en Python es la siguiente:


```
while <expresión>:
    <sentencia(s)>
```
  
`<sentencia(s)>` es el bloque de sentencias en Python que se ejecutará de manera indefinida hasta que no se cumpla la expresión.

`<expresión>` es una expresión evaluada en un contexto booleano. El resultado de evaluar esta expresión debe ser `True` o `False`. Controla la ejecución del bucle `while`.

In [ ]:
# El while repite MIENTRAS la condición sea verdadera. Si el cuerpo no
# modifica la variable de control, el bucle no termina nunca.
num = 10
while num > 0:
    num -= 1
    print(num)

In [ ]:
# El else de un while se ejecuta cuando el bucle termina de forma natural,
# es decir, cuando la condición deja de cumplirse sin que haya un break.
while num > 0:
    num -= 1
    print(num)
else:
    print("Sentencia final del bucle")

In [ ]:
# Todo en una línea es válido, pero se lee mal: en código de trabajo
# conviene el formato con sangría de la celda anterior.
num = 5
while num > 0: num -= 1; print(num)

## 4. Sentencia break

La sentencia break es una palabra reservada de Python que permite interrumpir de forma inmediata la ejecución de un bucle for o while. Cuando esta instrucción es alcanzada, el flujo de control abandona la estructura iterativa en curso y se transfiere directamente a la primera línea de código ubicada fuera del bloque correspondiente. Este mecanismo resulta especialmente útil para finalizar anticipadamente una iteración cuando se satisface una condición específica o cuando continuar con el bucle deja de ser necesario.

In [ ]:
colores = ["rojo", "verde", "azul"]
for color in colores:
    print(color)

In [ ]:
for color in colores:
    print(color)
    break

In [ ]:
# Aquí el break interrumpe el bucle en la primera vuelta, y por eso el else
# NO se ejecuta: esa es exactamente la diferencia con el caso anterior.
for color in colores:
    print(color)
    break
else:
    print("Ejecución en la claúsula else")

## 5. Sentencia `continue`

La sentencia continue es otra palabra reservada de Python que, a diferencia de break, no interrumpe por completo la ejecución de un bucle for o while. En lugar de ello, finaliza únicamente la iteración actual y transfiere el control al siguiente ciclo de evaluación del bucle. Este comportamiento resulta especialmente útil cuando se desea omitir determinadas condiciones o casos específicos sin detener por completo la estructura iterativa.

In [ ]:
colores = ["azul", "verde", "rojo"]
for color in colores:
    print(color)


In [ ]:
for color in colores:
    if color == "verde":
        continue
    print(color)

## 6. Sentencia pass

La sentencia pass es una palabra reservada de Python que permite definir la estructura sintáctica de bloques como funciones, sentencias if, bucles while o for, sin incorporar todavía instrucciones en su interior. Su función principal es actuar como una operación nula, es decir, una instrucción que no produce ningún efecto durante la ejecución, pero que permite mantener un bloque de código válido desde el punto de vista sintáctico. Esto resulta especialmente útil al diseñar prototipos, dejar implementaciones pendientes o establecer la estructura preliminar de un programa.

In [ ]:
for color in colores:
    # Aqui voy a hacer algo que todavia no se
    pass

---

# VII. Comprensiones: la escritura estándar en ciencia de datos

Todo lo que hizo con bucles en la sección anterior puede escribirse en una sola
expresión. No es un atajo estético: es la forma en que está escrito el código que
va a leer en documentación, ejemplos y proyectos reales.

```
[ expresión      for elemento in iterable      if condición ]
   qué guardo          qué recorro              qué filtro (opcional)
```

In [ ]:
notas = [3.2, 5.8, 4.0, 2.9, 6.5, 4.7]

# Con bucle: tres líneas y una lista que se va llenando
aprobadas_bucle = []
for n in notas:
    if n >= 4.0:
        aprobadas_bucle.append(n)

# Con comprensión: la misma lógica en una expresión
aprobadas = [n for n in notas if n >= 4.0]

print("Con bucle     :", aprobadas_bucle)
print("Con comprensión:", aprobadas)
assert aprobadas == aprobadas_bucle, "Ambas formas deben dar el mismo resultado"

### Las dos posiciones del `if`, que no significan lo mismo

Esta es la confusión que más errores silenciosos produce.

In [ ]:
# El if DESPUÉS del for: FILTRA. El resultado tiene menos elementos.
filtradas = [n for n in notas if n >= 4.0]

# El if ANTES del for: TRANSFORMA. El resultado conserva el largo original.
etiquetadas = ["aprobada" if n >= 4.0 else "reprobada" for n in notas]

print(f"Original  ({len(notas)} elementos): {notas}")
print(f"Filtrada  ({len(filtradas)} elementos): {filtradas}")
print(f"Etiquetada ({len(etiquetadas)} elementos): {etiquetadas}")

# Confundirlas cambia el largo del resultado, y el problema aparece varias
# celdas después, cuando ya no es evidente de dónde vino.
assert len(etiquetadas) == len(notas), "La forma con else conserva el largo"

In [ ]:
# La misma idea sirve para diccionarios y conjuntos
ciudades = ["Valparaiso", "Santiago", "Valparaiso", "Concepcion"]

longitudes = {c: len(c) for c in ciudades}          # comprensión de diccionario
unicas = {c.upper() for c in ciudades}              # comprensión de conjunto

print("Diccionario:", longitudes)
print("Conjunto   :", unicas, " <- sin repetidos, porque es un set")

# En el preprocesamiento esto aparece todo el tiempo: por ejemplo, para construir
# el mapa de codificación de una variable ordinal.
orden = ["bajo", "medio", "alto"]
mapa = {categoria: i for i, categoria in enumerate(orden)}
print("Mapa ordinal:", mapa)

> **Cuándo NO usar una comprensión.** Si necesita más de una condición anidada
> o el cuerpo no cabe cómodamente en una línea, vuelva al bucle: se lee mejor. La
> comprensión es para transformaciones simples, no para esconder complejidad.

## Ejercicio — sección VII

Con la misma lista, resuelva con **comprensiones** lo que antes hizo con bucles:

1. `positivas`: filtre los valores medidos y positivos.
2. `etiquetas`: para cada valor válido, `"alta"` si supera 140 y `"normal"` si no.

Preste atención a la posición del `if`: en la primera **filtra** y en la segunda
**transforma**.

In [ ]:
glucosas = [98.2, 110.5, None, 187.4, -5.0, 210.3, 142.0]

positivas = None   # TODO: comprensión con if DESPUÉS del for
etiquetas = None   # TODO: comprensión con if ANTES del for, sobre 'positivas'

print("Positivas:", positivas)
print("Etiquetas:", etiquetas)

assert positivas == [98.2, 110.5, 187.4, 210.3, 142.0]
assert len(etiquetas) == len(positivas), "La forma con else conserva el largo"
assert etiquetas == ["normal", "normal", "alta", "alta", "alta"]
print("Correcto")

<details>
<summary><b>Ver solución</b></summary>

```python
positivas = [g for g in glucosas if g is not None and g > 0]
etiquetas = ["alta" if g > 140 else "normal" for g in positivas]
```

</details>

## Ejercicio — sección VI

Recorra la lista de glucosas y construya dos resultados:

- `validas`: solo los valores medidos y positivos (descarte `None` y negativos).
- `alerta`: el **primer** valor sobre 200 que encuentre, o `None` si no hay ninguno.

Use `continue` para descartar y `break` para detenerse en la primera alerta.

In [ ]:
glucosas = [98.2, 110.5, None, 187.4, -5.0, 210.3, 142.0, 260.0]

validas = []
alerta = None

# TODO: complete el recorrido

print("Válidas:", validas)
print("Primera alerta:", alerta)

assert validas == [98.2, 110.5, 187.4, 210.3, 142.0, 260.0]
assert alerta == 210.3, "Debe ser el PRIMER valor sobre 200, no el último"
print("Correcto")

<details>
<summary><b>Ver solución</b></summary>

```python
for g in glucosas:
    if g is None or g < 0:
        continue                  # descarta y sigue con el siguiente
    validas.append(g)

for g in validas:
    if g > 200:
        alerta = g
        break                     # se detiene en la primera alerta
```

</details>

# VIII. Programación Modular


La programación modular constituye un principio fundamental en el diseño de software contemporáneo, especialmente en contextos donde la complejidad computacional, la mantenibilidad del código y la escalabilidad de las soluciones adquieren un rol central. Este enfoque propone descomponer un sistema en unidades funcionales más pequeñas, cohesionadas y relativamente independientes, denominadas módulos, cada una de las cuales asume una responsabilidad específica dentro de la solución global. En contraposición a una arquitectura monolítica, en la que la lógica del programa se concentra en un único bloque de código, la modularización permite estructurar el desarrollo en componentes que pueden ser diseñados, implementados, evaluados y reutilizados de manera separada, para luego integrarse en una arquitectura común y consistente. Desde esta perspectiva, la modularidad no solo favorece la claridad conceptual del sistema, sino que también mejora su capacidad de adaptación, evolución y trazabilidad a lo largo del ciclo de vida del software.

Desde un punto de vista técnico, la modularización ofrece múltiples beneficios. En primer lugar, contribuye a la simplicidad estructural, al permitir abordar problemas complejos mediante una estrategia de descomposición en subproblemas más acotados y manejables. En segundo lugar, facilita el mantenimiento del software, ya que una adecuada separación de responsabilidades reduce el acoplamiento entre componentes y, por tanto, minimiza el impacto de las modificaciones sobre el resto del sistema. Del mismo modo, potencia la reutilización, puesto que un módulo correctamente diseñado puede emplearse en distintos contextos o proyectos sin necesidad de replicar lógica previamente implementada. A ello se suma el control del espacio de nombres, dado que cada módulo define su propio namespace, lo que reduce la probabilidad de colisiones entre identificadores y favorece una organización semántica más rigurosa del código.

En Python, la modularidad se expresa a través de distintas construcciones del lenguaje que permiten articular soluciones de complejidad creciente. Las funciones encapsulan tareas específicas en bloques reutilizables; las clases organizan datos y comportamientos bajo una misma abstracción conceptual; los módulos estructuran el código en archivos independientes; y los paquetes agrupan múltiples módulos dentro de una jerarquía lógica más amplia. En conjunto, estas construcciones favorecen el desarrollo de sistemas más robustos, mantenibles y escalables, una condición particularmente relevante en proyectos de ciencia de datos e inteligencia artificial, donde los flujos de procesamiento suelen involucrar ingestión de datos, transformación, modelamiento, evaluación, visualización y despliegue, todos ellos susceptibles de beneficiarse de una arquitectura modular.

En este marco, resulta importante precisar que, en Python, un módulo puede presentarse de distintas formas: puede estar escrito directamente en Python, puede haber sido implementado en C y cargado dinámicamente durante la ejecución, o bien puede formar parte de los módulos integrados en el propio intérprete. No obstante, independientemente de su forma de implementación, el acceso a su contenido se realiza de manera uniforme mediante la instrucción import. En la práctica, el tipo de módulo más frecuente y pedagógicamente más relevante es el módulo escrito en Python, ya que constituye la forma más habitual de organizar código en el desarrollo cotidiano. Desde una perspectiva operativa, un módulo de este tipo no es más que un archivo con extensión .py que contiene definiciones de funciones, clases, variables y otras sentencias ejecutables. En consecuencia, cualquier archivo .py puede ser tratado como un módulo siempre que su contenido sea importado desde otro programa.

import modulo



Cuando el intérprete de Python procesa la instrucción import, localiza el módulo solicitado mediante una ruta de búsqueda compuesta por distintos directorios. En dicha ruta se incluyen el directorio de ejecución del script —o el directorio actual en modo interactivo—, los directorios definidos en PYTHONPATH, en caso de existir, y las rutas asociadas a la instalación del intérprete.

Esta ruta de búsqueda puede examinarse mediante la variable sys.path, definida en el módulo sys. Su contenido indica los directorios en los que Python intentará localizar módulos y paquetes al realizar una importación.

In [ ]:
import sys

In [ ]:
sys.path

Por lo tanto, para asegurarnos de que el módulo es encontrado, se requiere que hagamos alguna de las siguientes acciones:

* Guardar el `módulo.py` en el directorio donde se encuentra el script o en el directorio actual si estamos utilizando el interprete interactivo**
* Modificar la variable de entorno `PYTHONPATH` para que contenga el directorio donde se encuentra el módulo, o bien, guardar el módulo en uno de los directorios ya contenidos en esta variable
* Guardar el módulo en uno de los directorios dependientes de la instalación, a los que puede o no tener acceso de escritura, dependiendo del sistema operativo

### 1. `import`

La instrucción import permite incorporar un módulo dentro del programa actual para utilizar las funciones, clases, variables y demás elementos que estén definidos en él.

La forma más sencilla de emplearla es la siguiente:

In [ ]:
#import <nombre_modulo>

Para evitar tener que invocar todos los objetos definidos en el módulo utilizando el nombre del módulo, Python nos proporciona una sintaxis alternativa:
```
from <nombre_modulo> import <nombre(s)>
```
Esto nos permite importar cualquier objeto del módulo en el namespace del programa actual.

In [ ]:
%%writefile saludos.py
def hola():
    print("Hola")

In [ ]:
from saludos import hola

hola()

### 2. Alias

Una de las cosas interesantes que nos proporciona Python a nivel sintáctico cuando importamos elementos de un módulo es asignarles un alias. De esta manera, los objetos se almacenan en el namespace del programa con otro diferente al original. Para ello podemos utilizar la siguiente sintaxis:
```
import <nombre_modulo> as <nombre_alias>

from <nombre_modulo> import <nombre_objeto> as <nombre_alias>
```

In [ ]:
#Alias para un módulo
import math as m
print(m.sqrt(16))

In [ ]:
#Alias para un objeto del módulo
from math import sqrt as raiz
print(raiz(25))

### 3. Paquetes

Para que un directorio sea reconocido por Python como un paquete tradicional, normalmente debe contener un archivo llamado `__init__.py`. Este archivo indica que la carpeta forma parte de una estructura organizada de módulos que puede importarse desde otros programas.

El contenido de `__init__.py` se ejecuta la primera vez que el paquete es importado. Esto también ocurre cuando se importa alguno de sus módulos internos, ya que Python carga primero el paquete y luego el módulo solicitado. Por esta razón, este archivo puede utilizarse para realizar tareas de inicialización, organizar qué elementos del paquete estarán disponibles o preparar cierta configuración común.

De este modo, si un módulo se encuentra dentro de un paquete, para importarlo debe utilizarse una sintaxis que incluya tanto el nombre del paquete como el nombre del módulo.

```
import <nombre_paquete>.<nombre_modulo>

from <nombre_paquete>.<nombre_modulo> import <nombre(s)>
```

In [ ]:
from math import sqrt as raiz
print(raiz(25))

### 4. Importando todos los módulos de un paquete

Podríamos pensar que para importar todos los módulos de un paquete podríamos utilizar la sintaxis:
```
from <nombre_paquete> import *
```
Sin embargo, ¿qué es lo que sucede?

Python sigue la siguiente convención: si el archivo `__init__.py` en el directorio del paquete contiene una lista llamada `__all__`, se toma como una lista de módulos que deben ser importados cuando se encuentra la sentencia:
```
from <nombre_paquete> import *.
```

mi_paquete/
│
├── __init__.py
├── modulo1.py
└── modulo2.py

In [ ]:
import os

os.makedirs("mi_paquete", exist_ok=True)

In [ ]:
%%writefile mi_paquete/__init__.py
__all__ = ["modulo1", "modulo2"]

In [ ]:
%%writefile mi_paquete/modulo1.py
def saludar():
    print("Hola desde modulo1")

In [ ]:
%%writefile mi_paquete/modulo2.py
def despedir():
    print("Adiós desde modulo2")

In [ ]:
from mi_paquete import *

In [ ]:
from mi_paquete import modulo1, modulo2

modulo1.saludar()
modulo2.despedir()

#### a. Actualizando pip: >>> python -m pip install --upgrade pip



#### b. Instalando paquetes con pip: >>> pip install pandas

#### c. Listar los paquetes instalados en el entorno: >>> pip list

#### d. Mostrar los detalles de un paquete instalado: >>> pip show pandas

#### e. Exportando los paquetes instalados en el entorno: >>> pip freeze > requirements.txt

#### f. Instalando `requirements.txt`: >>> pip install -r requirements.txt

In [ ]:
#Instalar un paquete
!python -m pip install pandas

In [ ]:
!python -m pip list

In [ ]:
!python -m pip show pandas

In [ ]:
!python -m pip freeze > requirements.txt

#### g. Desinstalando paquetes: >>> pip uninstall pandas

---

# IX. Entorno reproducible y control de versiones

La unidad temática de esta semana declara el **control de versiones con git y
repositorios remotos** como contenido, y el indicador **ID 1.2** lo evalúa. Es lo
único del módulo que no se practica dentro del cuaderno, porque git opera sobre
archivos y carpetas, no sobre celdas.

Lo que sí puede hacer aquí es dejar el proyecto **en condiciones de ser
versionado**: declarar las dependencias y decidir qué archivos entran al
repositorio.

### Decidir qué NO se versiona también es una decisión técnica

Quedan fuera tres cosas: lo que se **regenera** (el entorno virtual, los datos
procesados), lo que es **local** de cada equipo (cachés, archivos del sistema) y
lo que es **secreto** (credenciales, tokens).

In [ ]:
CONTENIDO_GITIGNORE = """# Entorno virtual: se reconstruye con requirements.txt
.venv/

# Caché de Python y de Jupyter
__pycache__/
*.pyc
.ipynb_checkpoints/

# Archivos del sistema operativo
.DS_Store
Thumbs.db

# Credenciales: nunca al repositorio
.env
kaggle.json

# Datos procesados: se regeneran ejecutando el pipeline
data/processed/
"""

Path(".gitignore").write_text(CONTENIDO_GITIGNORE, encoding="utf-8")
print(".gitignore creado:\n")
print(CONTENIDO_GITIGNORE)

In [ ]:
# requirements.txt con las versiones EFECTIVAMENTE instaladas: declarar las
# supuestas es justamente lo que impide reconstruir el entorno.
paquetes = ["numpy", "pandas", "matplotlib", "scikit-learn", "jupyterlab"]

lineas = []
for nombre in paquetes:
    # scikit-learn se instala con ese nombre, pero se importa como 'sklearn'
    modulo = nombre.replace("scikit-learn", "sklearn")
    try:
        version = getattr(__import__(modulo), "__version__", None)
        lineas.append(f"{nombre}=={version}" if version else nombre)
    except ImportError:
        lineas.append(nombre)      # no instalado: se declara sin versión

Path("requirements.txt").write_text("\n".join(lineas) + "\n", encoding="utf-8")
print("requirements.txt:\n")
print(Path("requirements.txt").read_text(encoding="utf-8"))

### El ciclo de trabajo con git

Estos comandos se ejecutan en la **terminal**, no en el cuaderno. El detalle está
en la *Guía de Git y GitHub* del curso, con versión para Windows y para macOS.

```bash
git pull                       # 1. traer lo del equipo, ANTES de trabajar
# ... editar archivos ...
git status                     # 2. ver qué cambió
git add .                      # 3. preparar los cambios
git commit -m "feat: agrega limpieza de valores faltantes"
git pull                       # 4. traer de nuevo, por si alguien subió algo
git push                       # 5. subir
```

**Los tres estados.** Un archivo pasa por el *working directory* (usted edita),
el *staging area* (`git add`: elige qué entra en el registro) y el historial
(`git commit`: queda guardado con autor, fecha y mensaje).

> **Git no es GitHub.** Git funciona en su computador y sin internet; GitHub es
> un servicio que aloja repositorios git. Puede usar git sin GitHub, no al revés.

> **Lo que la rúbrica mira:** commits de **cada integrante**, repartidos en el
> tiempo y con mensajes descriptivos. Un repositorio con un solo autor y tres
> commits la noche anterior no alcanza el nivel superior, por completo que esté
> el contenido.

In [ ]:
# ¿Quedó el proyecto listo para versionar?
comprobaciones = {
    ".gitignore": Path(".gitignore").is_file(),
    "requirements.txt": Path("requirements.txt").is_file(),
    "data/raw/": Path("data/raw").is_dir(),
    "src/": Path("src").is_dir(),
}
for nombre, ok in comprobaciones.items():
    print(f"[{'OK  ' if ok else 'FALTA'}] {nombre}")

assert all(comprobaciones.values()), "Falta algún elemento del proyecto"
print("\nEl proyecto puede inicializarse con git init.")

## Ejercicio — sección VIII

Cree el módulo `src/limpieza.py` con dos funciones e impórtelas:

- `normalizar(texto)`: quita espacios sobrantes, colapsa los internos y pasa a minúsculas.
- `en_rango(valor, minimo, maximo)`: devuelve `True` si el valor está dentro del rango.

Use `%%writefile` en una celda aparte, como en la sección.

In [ ]:
from pathlib import Path
Path("src").mkdir(exist_ok=True)

# TODO: escriba el módulo. Puede hacerlo con Path.write_text o con %%writefile
# en una celda nueva. Aquí se muestra la primera forma:
codigo_modulo = """
TODO
"""
Path("src/limpieza.py").write_text(codigo_modulo, encoding="utf-8")

import importlib, sys
sys.path.insert(0, ".")
import src.limpieza as limpieza
importlib.reload(limpieza)

assert limpieza.normalizar("  Fumador   Activo ") == "fumador activo"
assert limpieza.en_rango(50, 0, 120) is True
assert limpieza.en_rango(150, 0, 120) is False
print("Correcto")

<details>
<summary><b>Ver solución</b></summary>

```python
codigo_modulo = '''
"""Utilidades de limpieza para el proyecto."""


def normalizar(texto):
    """Quita espacios sobrantes, colapsa los internos y pasa a minúsculas."""
    return " ".join(str(texto).split()).lower()


def en_rango(valor, minimo, maximo):
    """True si el valor está dentro del rango declarado."""
    return minimo <= valor <= maximo
'''
```

</details>

# X. Excepciones

En Python, los errores que aparecen durante la ejecución de un programa se gestionan mediante un mecanismo llamado excepciones. Una excepción es un objeto que representa una situación anómala o inesperada, como intentar dividir por cero, acceder a un elemento inexistente o abrir un archivo que no se encuentra disponible. Cuando ocurre uno de estos eventos, Python genera automáticamente una excepción e interrumpe el flujo normal de ejecución.

Si el programa incluye instrucciones para manejar esa situación, la excepción puede capturarse y tratarse de forma controlada, permitiendo que la ejecución continúe o que el sistema responda adecuadamente al problema. En cambio, si no existe un manejo explícito, el programa se detiene y Python muestra un mensaje con información sobre el error ocurrido. Desde una perspectiva de desarrollo de software, este mecanismo es esencial para construir aplicaciones más robustas, seguras y tolerantes a fallos.

En Python, las excepciones se gestionan principalmente mediante las sentencias try y except. El bloque try contiene el código que podría generar un error, mientras que el bloque except define cómo actuar si dicho error ocurre. De esta forma, el manejo de excepciones no solo permite detectar fallos, sino también controlar de manera más precisa el comportamiento del programa ante situaciones imprevistas.

### 1. Errores de sintaxis vs Excepciones

A pesar de que en muchos contextos estos términos suelen emplearse de manera indistinta, en Python es importante distinguir entre errores de sintaxis y excepciones, ya que corresponden a situaciones diferentes dentro del proceso de ejecución de un programa.

Los errores de sintaxis se producen cuando el código escrito no respeta las reglas formales del lenguaje. En estos casos, el problema no está en la lógica de ejecución, sino en que la instrucción simplemente no puede ser interpretada correctamente por Python. Cuando esto ocurre, el intérprete genera un mensaje identificado como SyntaxError y suele señalar la ubicación aproximada del problema mediante el carácter ^, facilitando así la detección del fragmento de código inválido.

A diferencia de las excepciones, los errores de sintaxis se detectan antes de que el programa pueda ejecutarse normalmente. Por ello, constituyen fallos en la escritura del código que deben corregirse para que la ejecución sea posible.

In [ ]:
# Demostración de error de SINTAXIS: falta la comilla de cierre de la cadena.
# Un error de sintaxis impide que Python interprete la celda, así que no
# puede capturarse con try/except: el código se pasa como texto a compile().
codigo = 'print("Hola mundo)'

try:
    compile(codigo, "<demostración>", "exec")
except SyntaxError as e:
    print(f"SyntaxError: {e.msg}")
    print(f"  en el código: {codigo}")

Los errores de sintaxis no pueden ser gestionados mediante mecanismos de control de excepciones, ya que no corresponden a excepciones propiamente tales dentro del lenguaje. Estos errores se producen cuando el código escrito no respeta las reglas formales de la sintaxis de Python, por lo que el intérprete no puede siquiera comenzar a ejecutarlo correctamente. En cambio, las excepciones se originan durante la ejecución de sentencias que sí son sintácticamente válidas, pero que generan una situación anómala, como una división por cero o el intento de acceder a un recurso inexistente. A diferencia de los errores de sintaxis, las excepciones no necesariamente implican la detención definitiva del programa, ya que pueden ser capturadas, gestionadas o incluso ignoradas mediante las estructuras de control apropiadas.

### 2. Gestión de excepciones: Sentencias `try` y `except`

Las sentencias `try` y `except` en Python pueden utilizarse para capturar y manejar excepciones. Python ejecuta el código que sigue a la sentencia `try` como una parte "normal" del programa. El código que sigue a la sentencia `except` se ejecutará si se produce cualquier excepción en la cláusula `try` precedente.

La sintaxis que se utiliza para definir este tipo de comportamiento es la siguiente:
```
try:
    <sentencia(s)>
except <excepción>:
    <sentencias(s) si excepción>
```

In [ ]:
# Demostración de error: la variable no está definida en este ámbito.
# Se ejecuta dentro de try/except para que el cuaderno pueda correr
# completo; el mensaje que imprime es exactamente el que verían.
try:
    print(var)
except NameError as e:
    print(f"NameError: {e}")

In [ ]:
try:
    print(var)
except NameError:
    var = "Hola mundo"

In [ ]:
# Demostración de error: la variable no está definida en este ámbito.
# Se ejecuta dentro de try/except para que el cuaderno pueda correr
# completo; el mensaje que imprime es exactamente el que verían.
try:
    print(var)
except NameError as e:
    print(f"NameError: {e}")

Debemos tener en cuenta que, si dentro del bloque try se produce una excepción, las sentencias ubicadas después de la instrucción que la genera no llegarán a ejecutarse. En ese caso, el flujo del programa se transfiere inmediatamente al bloque except correspondiente.

In [ ]:
# El try/except no detiene el programa: lo que sigue al bloque se ejecuta igual.
# Fíjese en que la línea posterior al error, DENTRO del try, sí se salta.
print("Sentencias de codigo antes del try")

try:
    print("Codigo antes de la excepcion")
    10 + "3"
    print("Codigo despues de la excepcion")
except TypeError:
    print("No se puede sumar un numero entero y un string")

print("Sentencias de codigo despues del except")

También podemos utilizar la sentencia `except` sin indicarle el nombre de ninguna excepción, en estos casos capturará todas las excepciones que se produzcan en el código que se encuentra en el cuerpo de la sentencia `try`

In [ ]:
try:
    200 + "10"
except:
    print("No sumar un entero con un string")

Además de la sintaxis básica, es posible capturar distintos tipos de excepciones mediante el uso de múltiples cláusulas except. Esto permite definir un tratamiento específico para cada tipo de error, lo que proporciona un mayor control sobre el comportamiento del programa ante distintas situaciones excepcionales.

In [ ]:
# Se pueden encadenar varios except: Python entra en el primero que coincida
# con el tipo de error ocurrido. Los demás se ignoran.
try:
    50/0
except NameError:
    print("Gestionando excepcion NameError")
except TypeError:
    print("Gestionando execpcion Type Error")
except ZeroDivisionError:
    print("No puedes dividir un numero por 0")

Finalmente, Python permite asociar el objeto de excepción capturado a una variable, lo que hace posible acceder a su contenido y utilizarlo para proporcionar información más específica sobre el error producido.

In [ ]:
# 'as error' captura el objeto de la excepción, que trae el tipo y el mensaje.
# Es lo que permite informar con precisión en vez de un aviso genérico.
try:
    print(variable)
except NameError as error:
    print("Objeto de tipo:", type(error))
    print("La excepcion consiste en:", error)

## Ejercicio — sección X

Escriba `a_numero(texto)` que convierta una cadena a `float` de forma segura:

- Acepta el punto o la **coma** como separador decimal.
- Devuelve `None` si el texto no representa un número, en lugar de detener el programa.

Los archivos reales traen `"110,5"` con coma decimal: si no se contempla, la
conversión falla y se pierde un dato válido.

In [ ]:
def a_numero(texto):
    """TODO: convertir a float admitiendo coma decimal; None si no se puede."""
    pass


valores = ["110.5", "98,2", "sin dato", "", "187.4"]
convertidos = [a_numero(v) for v in valores]
print(convertidos)

assert convertidos == [110.5, 98.2, None, None, 187.4]
print("Correcto: la coma decimal se convierte y lo inválido no rompe el proceso")

<details>
<summary><b>Ver solución</b></summary>

```python
def a_numero(texto):
    """Convierte a float admitiendo coma decimal. Devuelve None si no es número."""
    try:
        return float(str(texto).replace(",", "."))
    except (ValueError, TypeError):
        return None
```

</details>

# XI. Librerias para Ciencia de Datos

Las librerías para Ciencia de Datos son conjuntos de herramientas de software que amplían las capacidades de Python para trabajar de manera eficiente con datos, modelos y procesos analíticos. Estas librerías permiten realizar tareas fundamentales como la carga, limpieza, transformación, análisis, visualización y modelamiento de datos, proporcionando funciones y estructuras especializadas que simplifican problemas que, de otro modo, requerirían una gran cantidad de código. En este contexto, bibliotecas como NumPy facilitan el cálculo numérico y la manipulación de arreglos multidimensionales; pandas ofrece estructuras tabulares para el análisis de datos; Matplotlib y Seaborn permiten la visualización de información; y scikit-learn proporciona algoritmos de aprendizaje automático para clasificación, regresión, agrupamiento y evaluación de modelos. En conjunto, estas herramientas constituyen la base del ecosistema de ciencia de datos en Python y permiten desarrollar flujos de trabajo más robustos, reproducibles y escalables.

### 1. NumPy

NumPy constituye una de las bibliotecas fundamentales del ecosistema científico de Python, debido a que proporciona una infraestructura eficiente para la representación y el procesamiento de datos numéricos. Su núcleo está basado en el objeto ndarray, una estructura de arreglos multidimensionales que permite realizar operaciones vectorizadas de alto rendimiento, reduciendo la necesidad de iteraciones explícitas y mejorando significativamente la eficiencia computacional. Sobre esta base, NumPy incorpora un conjunto amplio de funciones matemáticas y numéricas para tareas como álgebra lineal, estadística básica, transformaciones de arreglos y generación de números aleatorios, todas ellas esenciales en el análisis de datos, la modelación matemática y el desarrollo de algoritmos de aprendizaje automático.

Desde una perspectiva aplicada, NumPy no solo facilita el trabajo con grandes volúmenes de datos numéricos, sino que también actúa como una capa base sobre la cual se construyen numerosas bibliotecas del ecosistema de ciencia de datos, como pandas, scikit-learn y SciPy. Adicionalmente, su capacidad de interoperar con código escrito en C, C++ y Fortran permite combinar productividad en Python con eficiencia de bajo nivel, una característica especialmente relevante en contextos de cómputo intensivo. En consecuencia, el dominio de NumPy resulta central para comprender y desarrollar flujos de trabajo robustos en ciencia de datos e inteligencia artificial.


In [ ]:
# Instalación de Numpy y Matplotlib
!pip install numpy

In [ ]:
import numpy as np

### 1.1 Array

Un array es una estructura de datos que permite almacenar múltiples elementos del mismo tipo de forma organizada. En NumPy, los arrays pueden tener una o varias dimensiones, lo que los convierte en una herramienta central para representar vectores, matrices y estructuras de datos más complejas. Cada posición dentro del array se identifica mediante uno o más índices, lo que permite acceder de manera precisa a sus elementos. Esta forma de organización resulta especialmente útil en ciencia de datos, ya que facilita operaciones numéricas eficientes sobre grandes volúmenes de información.

En el contexto de NumPy, cada dimensión del array se denomina eje (axis). El número total de dimensiones se conoce como número de dimensiones o rank. La forma (shape) corresponde a una tupla que indica cuántos elementos hay en cada dimensión, mientras que el tamaño (size) representa el número total de elementos almacenados en el array. Estas propiedades permiten describir con precisión la estructura del arreglo y comprender cómo se distribuyen sus datos.

In [ ]:
# ndim = número de dimensiones (aquí 2), shape = filas y columnas,
# size = total de elementos. Son los tres atributos que hay que mirar
# antes de operar con un array.
import numpy as np

a = np.array([[1, 2, 3],
              [4, 5, 6]])

print(a)
print("Número de dimensiones:", a.ndim)
print("Forma:", a.shape)
print("Tamaño:", a.size)

In [ ]:
# Array cuyos valores son todos 0
a = np.zeros((2, 4))

En este ejemplo, el array tiene dos ejes: uno correspondiente a las filas y otro a las columnas. Su rank es 2, porque posee dos dimensiones. Su shape es (2, 3), lo que indica que tiene 2 filas y 3 columnas. Finalmente, su size es 6, porque contiene 6 elementos en total.

Una forma breve de resumirlo sería esta:

* Axis: cada dimensión del array

* Rank / ndim: cantidad de dimensiones

* Shape: tamaño de cada dimensión

* Size: número total de elementos

In [ ]:
# Array cuyos valores son todos 0
a = np.zeros((2, 4))

In [ ]:
a

_**a**_ es un array:
* Con dos **axis**, el primero de longitud 2 y el segundo de longitud 4
* Con un **rank** igual a 2
* Con un **shape** igual (2, 4)
* Con un **size** igual a 8

In [ ]:
a.shape

In [ ]:
a.ndim

In [ ]:
a.size

### 1.2 Creación de arreglos en NumPy

La creación de arreglos en NumPy es fundamental para trabajar con datos numéricos de manera eficiente. A través del objeto ndarray, es posible representar vectores, matrices y estructuras multidimensionales sobre las cuales se pueden realizar operaciones de alto rendimiento. Los arreglos pueden generarse a partir de listas o tuplas, o bien inicializarse con funciones específicas como zeros, ones, arange o linspace, según las necesidades del problema. En ciencia de datos, comprender cómo construir correctamente estas estructuras es esencial, ya que constituye la base para el procesamiento, análisis y modelamiento de datos.

In [ ]:
# Array cuyos valores son todos 0
np.zeros((2, 3, 4))

In [ ]:
# Array cuyos valores son todos 1
np.ones((2, 3, 4))

In [ ]:
# Array cuyos valores son todos el valor indicado como segundo parámetro de la función
np.full((2, 3, 4), 8)

In [ ]:
# El resultado de np.empty no es predecible
# Inicializa los valores del array con lo que haya en memoria en ese momento
np.empty((2, 3, 9))

In [ ]:
# Inicializacion del array utilizando un array de Python
b = np.array([[1, 2, 3], [4, 5, 6]])
b

In [ ]:
b.shape

In [ ]:
# Creación del array utilizando una función basada en rangos
# (minimo, maximo, número elementos del array)
print(np.linspace(0, 6, 10))

In [ ]:
# Inicialización del array con valores aleatorios
np.random.rand(2, 3, 4)

In [ ]:
# Inicialización del array con valores aleatorios conforme a una distribución normal
np.random.randn(2, 4)

### 1.3 Indexación y acceso a los elementos de un array

En NumPy, los elementos de un array se acceden mediante índices, los cuales indican la posición de cada valor dentro de la estructura. En arreglos multidimensionales, se utiliza un índice por cada dimensión. Además, NumPy permite extraer subconjuntos de datos mediante segmentación (slicing), lo que facilita seleccionar filas, columnas o intervalos específicos de forma eficiente. Estas operaciones son esenciales para manipular y analizar datos numéricos.

In [ ]:
# Creación de un Array unidimensional
array_uni = np.array([1, 3, 5, 7, 9, 11])
print("Shape:", array_uni.shape)
print("Array_uni:", array_uni)

In [ ]:
# Accediendo al quinto elemento del Array
array_uni[4]

In [ ]:
# Accediendo al tercer y cuarto elemento del Array
array_uni[2:4]

In [ ]:
# Accediendo a los elementos 0, 3 y 5 del Array
array_uni[0::3]

In [ ]:
# Creación de un Array multidimensional
array_multi = np.array([[1, 2, 3, 4], [5, 6, 7, 8]])
print("Shape:", array_multi.shape)
print("Array_multi:\n", array_multi)

In [ ]:
# Accediendo al cuarto elemento del Array
array_multi[0, 3]

In [ ]:
# Accediendo a la segunda fila del Array
array_multi[1, :]

In [ ]:
# Accediendo al tercer elemento de las dos primeras filas del Array
array_multi[0:2, 2]

In [ ]:
# Creación de un Array unidimensional inicializado con el rango de elementos 0-27
array1 = np.arange(28)
print("Shape:", array1.shape)
print("Array 1:", array1)

In [ ]:
# Cambiar las dimensiones del Array y sus longitudes
array1.shape = (7, 4)
print("Shape:", array1.shape)
print("Array 1:\n", array1)

In [ ]:
# El ejemplo anterior devuelve un nuevo Array que apunta a los mismos datos.
# Importante: Modificaciones en un Array, modificaran el otro Array
array2 = array1.reshape(4, 7)
print("Shape:", array2.shape)
print("Array 2:\n", array2)

In [ ]:
# Modificación del nuevo Array devuelto
array2[0, 3] = 20
print("Array 2:\n", array2)

In [ ]:
print("Array 1:\n", array1)

In [ ]:
# Desenvuelve el Array, devolviendo un nuevo Array de una sola dimension
# Importante: El nuevo array apunta a los mismos datos
print("Array 1:", array1.ravel())

In [ ]:
# Creación de dos Arrays unidimensionales
array1 = np.arange(2, 18, 2)
array2 = np.arange(8)
print("Array 1:", array1)
print("Array 2:", array2)

In [ ]:
# Suma
print(array1 + array2)

In [ ]:
# Resta
print(array1 - array2)

In [ ]:
# Multiplicacion
# Importante: No es una multiplicación de matrices
print(array1 * array2)

### 1.4 Broadcasting en NumPy

El broadcasting es el mecanismo mediante el cual NumPy permite realizar operaciones entre arreglos de distintas dimensiones o tamaños, siempre que sus formas sean compatibles. En lugar de replicar explícitamente los datos para igualar sus dimensiones, NumPy ajusta internamente la forma de los arreglos siguiendo ciertas reglas, lo que permite ejecutar operaciones vectorizadas de manera más eficiente en memoria y tiempo de cómputo. Este mecanismo resulta especialmente útil en ciencia de datos y computación numérica, ya que facilita operaciones entre vectores, matrices y escalares sin necesidad de escribir bucles explícitos.


In [ ]:
# BROADCASTING: el array b, de una fila, se 'estira' automáticamente para
# operar con cada fila de a. NumPy no copia datos: ajusta las formas.
import numpy as np

a = np.array([[1, 2, 3],
              [4, 5, 6]])

b = np.array([10, 20, 30])

print(a + b)

In [ ]:
# Creación de dos Arrays unidimensionales
array1 = np.arange(5)
array2 = np.array([3])
print("Shape Array 1:", array1.shape)
print("Array 1:", array1)
print()
print("Shape Array 2:", array2.shape)
print("Array 2:", array2)

In [ ]:
# Suma de ambos Arrays
array1 + array2

In [ ]:
# Creación de dos Arrays multidimensional y unidimensional
array1 = np.arange(6)
array1.shape = (2, 3)
array2 = np.arange(6, 18, 4)
print("Shape Array 1:", array1.shape)
print("Array 1:\n", array1)
print()
print("Shape Array 2:", array2.shape)
print("Array 2:", array2)

In [ ]:
# Suma de ambos Arrays
array1 + array2

### 1.5 Funciones estadísticas sobre arrays en NumPy

NumPy incorpora un conjunto de funciones estadísticas que permiten resumir y analizar la distribución de los datos contenidos en un array de forma eficiente. Entre las más utilizadas se encuentran el cálculo de la media, la mediana, la desviación estándar, la varianza, el mínimo y el máximo. Estas funciones pueden aplicarse sobre el arreglo completo o sobre un eje específico, lo que permite obtener métricas descriptivas tanto globales como por filas o columnas. En ciencia de datos, estas operaciones resultan fundamentales para la exploración inicial de datos, la caracterización de variables y la identificación de patrones básicos en conjuntos numéricos.


In [ ]:
# Sin el parámetro axis, estas funciones operan sobre TODOS los elementos.
# Con axis=0 el cálculo se hace por columna y con axis=1, por fila.
import numpy as np

a = np.array([[1, 2, 3],
              [4, 5, 6]])

print("Media:", np.mean(a))
print("Suma:", np.sum(a))
print("Mínimo:", np.min(a))
print("Máximo:", np.max(a))
print("Desviación estándar:", np.std(a))

### 2. Pandas

Pandas es una biblioteca fundamental del ecosistema de ciencia de datos en Python, orientada a la manipulación, transformación y análisis eficiente de datos estructurados. Su principal fortaleza radica en proporcionar estructuras de datos de alto rendimiento y de uso flexible, diseñadas para facilitar el tratamiento de información tabular en contextos analíticos y computacionales complejos.

La estructura central de Pandas es el DataFrame, que puede entenderse como una tabla bidimensional en memoria, organizada en filas y columnas, similar a una hoja de cálculo o a una tabla de base de datos relacional. Esta estructura permite trabajar con datos heterogéneos, incorporar etiquetas en filas y columnas, y ejecutar operaciones avanzadas de selección, filtrado, agregación y transformación. Asimismo, Pandas ofrece funcionalidades comparables a muchas de las operaciones disponibles en herramientas como Excel, incluyendo la generación de tablas dinámicas, el cálculo de nuevas columnas a partir de variables existentes, la agrupación de datos y su representación gráfica.

Desde una perspectiva aplicada, Pandas destaca por su capacidad para manipular grandes volúmenes de datos de forma eficiente, permitiendo operaciones como unión, segmentación, reorganización, limpieza y reestructuración de conjuntos de datos. Por ello, constituye una herramienta esencial en flujos de trabajo de ciencia de datos, ya que actúa como una capa intermedia entre la adquisición de datos, su preprocesamiento y su posterior análisis o modelamiento.

In [ ]:
# Instalación de Pandas
!pip install pandas
!pip install pyarrow

In [ ]:
import pandas as pd

### 2.1 Estructuras de datos en Pandas

En la práctica, Pandas se centra principalmente en dos estructuras de datos fundamentales: Series y DataFrame. La estructura Series representa una colección unidimensional de datos indexados, mientras que DataFrame corresponde a una tabla bidimensional compuesta por filas y columnas. Aunque antiguamente existió la estructura Panel para datos tridimensionales, esta fue eliminada de la biblioteca, por lo que ya no forma parte de su uso actual.

In [ ]:
# Creacion de un objeto Series
s = pd.Series([2, 4, 6, 8, 10])
print(s)

In [ ]:
# Creación de un objeto Series inicializándolo con un diccionario de Python
altura = {"Santiago": 187, "Pedro": 178, "Julia": 170, "Ana": 165}
s = pd.Series(altura)
print(s)

In [ ]:
# Creación de un objeto Series inicializándolo con algunos
# de los elementos de un diccionario de Python
altura = {"Santiago": 187, "Pedro": 178, "Julia": 170, "Ana": 165}
s = pd.Series(altura, index = ["Pedro", "Julia"])
print(s)

In [ ]:
# Creación de un objeto Series inicializandolo con un escalar
s = pd.Series(34, ["test1", "test2", "test3"])
print(s)

### 2.2 Acceso a los elementos de un objeto Series

En Pandas, el acceso a los elementos de un objeto Series se realiza mediante su índice, lo que permite recuperar valores de forma flexible y precisa. Dado que una Series es una estructura unidimensional compuesta por datos y etiquetas asociadas, sus elementos pueden consultarse tanto por posición como por etiqueta. Esta característica facilita la selección de valores individuales o subconjuntos de datos, y resulta especialmente útil en tareas de análisis y preprocesamiento, donde es frecuente trabajar con observaciones específicas dentro de una colección indexada.

In [ ]:
# Creación de un objeto Series
s = pd.Series([2, 4, 6, 8], index=["num1", "num2", "num3", "num4"])
print(s)

In [ ]:
# Accediendo al tercer elemento del objeto
s["num3"]

In [ ]:
# El acceso directo s[2] es AMBIGUO cuando el índice es de texto: pandas no
# sabe si 2 es una etiqueta o una posición, y su comportamiento ha cambiado
# entre versiones. Por eso se usan los accesores explícitos:
try:
    print('s[2]      ->', s[2])
except KeyError as e:
    print('s[2]      -> KeyError:', e, '(no existe la ETIQUETA 2)')

print('s.iloc[2] ->', s.iloc[2], '  <- por POSICIÓN, siempre correcto')
print('s.loc[...] ->', s.loc[s.index[2]], '  <- por ETIQUETA')

In [ ]:
# loc es la forma estándar de acceder a un elemento de un objeto Series por atributo
s.loc["num3"]

In [ ]:
# iloc es la forma estándar de acceder a un elemento de un objeto Series por posición
s.iloc[2]

In [ ]:
# Accediendo al segundo y tercer elemento por posición
s.iloc[2:4]

### 2.3 Creación de un objeto DataFrame

In [ ]:
# Creación de un DataFrame inicializándolo con un diccionario de objetios Series
personas = {
    "peso": pd.Series([84, 90, 56, 64], ["Santiago","Pedro", "Ana", "Julia"]),
    "altura": pd.Series({"Santiago": 187, "Pedro": 178, "Julia": 170, "Ana": 165}),
    "hijos": pd.Series([2, 3], ["Pedro", "Julia"])
}

df = pd.DataFrame(personas)
df

In [ ]:
# Creación de un DataFrame inicializándolo con una lista de listas de Python
# Importante: Deben especificarse las columnas e indices por separado
valores = [
    [185, 4, 76],
    [170, 0, 65],
    [190, 1, 89]
]

df = pd.DataFrame(
        valores,
        columns = ["altura", "hijos", "peso"],
        index = ["Pedro", "Ana", "Juan"])
df

In [ ]:
# Creación de un DataFrame inicializándolo con un diccionario de Python
personas = {
    "altura": {"Santiago": 187, "Pedro": 178, "Julia": 170, "Ana": 165},
    "peso": {"Santiago": 87, "Pedro": 78, "Julia": 70, "Ana": 65}}

df = pd.DataFrame(personas)
df

### 2.4 Acceso a los elementos de un DataFrame

In [ ]:
# Creación de un DataFrame inicializándolo con un diccionario de objetios Series
personas = {
    "peso": pd.Series([84, 90, 56, 64], ["Santiago","Pedro", "Ana", "Julia"]),
    "altura": pd.Series({"Santiago": 187, "Pedro": 178, "Julia": 170, "Ana": 165}),
    "hijos": pd.Series([2, 3], ["Pedro", "Julia"])
}

df = pd.DataFrame(personas)
df

In [ ]:
# Acceso a los elementos de las columnas del DataFrame
df["peso"]

In [ ]:
df[["peso", "altura"]]

In [ ]:
# Pueden combinarse los metodos anteriores con expresiones booleanas
df["peso"] > 80

In [ ]:
# Pueden combinarse los metodos anteriores con expresiones booleanas
df[df["peso"] > 80]

In [ ]:
# Acceso a los elementos de las filas del DataFrame
df.loc["Pedro"]

In [ ]:
df.iloc[2]

In [ ]:
df.iloc[1:3]

In [ ]:
# Consulta avanzada de los elementos de un DataFrame
df.query("altura >= 170 and peso > 60")

In [ ]:
# Copiar un DataFrame
# Creación de un DataFrame inicializándolo con un diccionario de objetios Series
personas = {
    "peso": pd.Series([84, 90, 56, 64], ["Santiago","Pedro", "Ana", "Julia"]),
    "altura": pd.Series({"Santiago": 187, "Pedro": 178, "Julia": 170, "Ana": 165}),
    "hijos": pd.Series([2, 3], ["Pedro", "Julia"])
}

df = pd.DataFrame(personas)
df

In [ ]:
# Copia del DataFrame df en df_copy
# Importante: Al modificar un elemento de df_copy no se modifica df
df_copy = df.copy()
df_copy

In [ ]:
# Añadir una nueva columna al DataFrame
df["cumpleaños"] = [1990, 1987, 1980, 1994]
df

In [ ]:
# Añadir una nueva columna calculada al DataFrame
df["años"] = 2020 - df["cumpleaños"]
df

In [ ]:
# Añadir una nueva columna creando un DataFrame nuevo
df_mod = df.assign(mascotas = [1, 3, 0, 0])
df_mod

In [ ]:
# Eliminar una columna existente del DataFrame
del df["peso"]
df

In [ ]:
# Eliminar una columna existente devolviendo una copia del DataFrame resultante
df_mod = df.drop(["hijos"], axis=1)
df_mod

2.5 Acceso a los elementos de un DataFrame

In [ ]:
# Creación de un DataFrame inicializándolo con un diccionario de objetios Series
personas = {
    "peso": pd.Series([84, 90, 56, 64], ["Santiago","Pedro", "Ana", "Julia"]),
    "altura": pd.Series({"Santiago": 187, "Pedro": 178, "Julia": 170, "Ana": 165}),
    "hijos": pd.Series([2, 3], ["Pedro", "Julia"])
}

df = pd.DataFrame(personas)
df

In [ ]:
# Guardar el DataFrame como CSV, HTML y JSON
df.to_csv("df_personas.csv")
df.to_html("df_personas.html")
df.to_json("df_personas.json")

In [ ]:
# Cargar el DataFrame en Jupyter
df2 = pd.read_csv("df_personas.csv")
df2

In [ ]:
# Cargar el DataFrame con la primera columna correctamente asignada
df2 = pd.read_csv("df_personas.csv", index_col=0)
df2


## Ejercicio — sección XI

Con el conjunto clínico ya cargado, obtenga con **pandas**:

1. `mayores`: las filas de pacientes de 65 años o más.
2. `imc_por_trabajo`: el IMC promedio por `work_type`, redondeado a dos decimales.
3. `pct_faltantes_bmi`: el porcentaje de valores faltantes en `bmi`.

In [ ]:
import pandas as pd
datos = pd.read_csv("data/raw/healthcare-dataset-stroke-data.csv")

mayores = None            # TODO
imc_por_trabajo = None    # TODO
pct_faltantes_bmi = None  # TODO

print("Pacientes de 65+ :", len(mayores))
print("\nIMC promedio por tipo de trabajo:\n", imc_por_trabajo)
print(f"\nFaltantes en bmi: {pct_faltantes_bmi:.1f}%")

assert (mayores["age"] >= 65).all(), "El filtro no está correcto"
assert isinstance(imc_por_trabajo, pd.Series) and len(imc_por_trabajo) > 1
assert 0 < pct_faltantes_bmi < 100
print("\nCorrecto")

<details>
<summary><b>Ver solución</b></summary>

```python
mayores = datos[datos["age"] >= 65]
imc_por_trabajo = datos.groupby("work_type")["bmi"].mean().round(2)
pct_faltantes_bmi = datos["bmi"].isna().mean() * 100
```

</details>

# XII. Preprocesamiento de datos

El preprocesamiento de datos es una etapa fundamental en ciencia de datos e inteligencia artificial, ya que condiciona la calidad de los análisis y modelos que se desarrollan posteriormente. Su finalidad es transformar datos brutos, incompletos o heterogéneos en representaciones más consistentes, comparables y adecuadas para su tratamiento computacional. Este proceso incluye tareas como el manejo de valores faltantes, la corrección de inconsistencias, la detección de observaciones atípicas, la codificación de variables y la normalización de atributos. En la práctica, estas transformaciones suelen organizarse en un pipeline, es decir, en una secuencia ordenada y reproducible de pasos aplicada sobre los datos. Un preprocesamiento bien diseñado no solo mejora el rendimiento de los modelos, sino que también fortalece la validez, robustez y reproducibilidad del proceso analítico.

### 1. Carga del conjunto de datos

La carga del conjunto de datos constituye una etapa inicial y esencial dentro del flujo de trabajo en ciencia de datos, ya que permite incorporar la información desde una fuente externa hacia un entorno computacional apto para su exploración, transformación y análisis. Este proceso implica la lectura de datos desde distintos formatos —como archivos CSV, Excel, bases de datos o servicios web— y su posterior representación en estructuras manipulables, como un DataFrame en Pandas. Desde una perspectiva metodológica, esta fase no se limita a una operación técnica de importación, sino que establece las condiciones de acceso, integridad y trazabilidad de la información sobre la cual se desarrollarán las etapas posteriores del proceso analítico.

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('data/raw/dat.csv')
display(df.head())

In [ ]:
df1 = pd.read_csv('data/raw/dat.csv',delimiter='|')
display(df1.head())

In [ ]:
import csv

In [ ]:
with open('data/raw/dat.csv') as file:
    reader = csv.reader(file,delimiter='|')
    for row in reader:
        print(row)

In [ ]:
df1 = pd.read_csv('data/raw/arch1.txt', delimiter=',', skiprows=2)
display(df1.head())

In [ ]:
df2 = pd.read_csv('data/raw/arch2.txt', delimiter=';', skiprows=2)
display(df2.head())

In [ ]:
df3 = pd.read_csv('data/raw/arch3.txt', delimiter='\t', skiprows=3)
display(df3.head())

In [ ]:
df4 = pd.read_csv('data/raw/arch4.txt', delimiter='|', skiprows=2)
display(df4.head())

In [ ]:
df5 = pd.read_csv('data/raw/arch5.txt', delimiter=';;', skiprows=2, engine='python')
display(df5.head())

In [ ]:
df = pd.read_excel('data/raw/dat_ejemplo.xlsx', sheet_name='Hoja1', usecols="A:B")
display(df.head())
df = pd.read_excel('data/raw/dat_ejemplo.xlsx', sheet_name='Hoja1')
display(df.head())

In [ ]:
# Un DataFrame se construye desde un diccionario: cada clave es una columna
# y cada lista, sus valores. Todas las listas deben tener el mismo largo.
data = {
    'Nombre': ['Juan', 'Ana', 'Luis', 'María'],
    'Edad': [28, 22, 35, 29],
    'Ciudad': ['Madrid', 'Barcelona', 'Valencia', 'Sevilla']
}
df = pd.DataFrame(data)
display(df.head())

In [ ]:
df.to_excel('data/raw/dataframe_ejemplo.xlsx', index=False)

In [ ]:
df = pd.read_csv("data/raw/datos.txt",delimiter='|')
display(df.head())

In [ ]:
display(df.tail())

In [ ]:
print(df.info())

In [ ]:
print(df.describe())

In [ ]:
print(df.dtypes)

In [ ]:
print(df['edad'].unique())


In [ ]:
print(df['ciudad'].value_counts())

### 2. Datos faltantes

Los datos faltantes corresponden a valores ausentes en una o más variables de un conjunto de datos y constituyen un problema metodológicamente relevante en ciencia de datos e inteligencia artificial. Su presencia puede alterar la distribución de las variables, reducir la potencia analítica de los modelos e introducir sesgos en los resultados si no se abordan de manera adecuada. Estos valores ausentes pueden originarse por errores de medición, fallas en la recolección, problemas de integración entre fuentes o ausencia efectiva de información en ciertas observaciones. Desde una perspectiva analítica, el tratamiento de datos faltantes no debe entenderse como una simple tarea de limpieza, sino como una decisión que afecta directamente la calidad de la representación de los datos y, por tanto, la validez, robustez e interpretabilidad del proceso de modelamiento.

**Usaremos un dataset péqueño**

Como se darán cuenta, el valor **np.nan** es una forma común de reflejar la presencia de valores perdidos o missing values (MV).

No siempre es asi. Como vimos en la clase teórica, a veces se pueden encontrar otros símbolos como: espacios en blanco, signos de interrogación, etc.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
datos = {'alumno': ['Zutano', 'Mengano', 'Zutano','Pepe' ,'Fulanito, Cosme', 'Maria', 'Juan', 'Pablo', 'Ana'],
         'Primera prueba': [np.nan, 8.5,7,7,np.nan, 4.5, 7.5, 1.0, 8.8],
         'Segunda prueba': [10,np.nan,9.8,4,np.nan,np.nan, 2.5, np.nan, np.nan],
         'observaciones':['ninguna','libre','ninguna','ninguna','libre','oyente','ninguna','libre','oyente'],
         'DNI':[23000000, 12389100,23000000, 99999, 1001,30406011, 1230000, 86758444, 9996696]}
df = pd.DataFrame(datos)
df

### 2.1 Eliminación de filas o columnas con valores faltantes (NaN)

El uso de df.dropna() presenta la ventaja de ser una estrategia simple, rápida y fácil de implementar, ya que permite trabajar únicamente con observaciones completas sin introducir supuestos adicionales sobre los valores faltantes. Sin embargo, su principal limitación es que puede generar una pérdida importante de información, especialmente cuando la proporción de datos faltantes es elevada o se concentra en variables relevantes. Además, al reducir el tamaño muestral, puede afectar la representatividad del conjunto de datos e introducir sesgos si la ausencia de valores no ocurre de manera aleatoria.

In [ ]:
df_2 = df.dropna()
df_2

### 2.2 Eliminación de columnas con valores faltantes mediante dropna(axis=1)

La eliminación de columnas con valores faltantes mediante dropna(axis=1) tiene la ventaja de ser una estrategia simple y directa, ya que permite conservar únicamente las variables completas, facilitando el análisis posterior y evitando la incorporación de supuestos adicionales sobre los datos ausentes. Sin embargo, su principal desventaja es que puede implicar una pérdida considerable de información, especialmente cuando las columnas eliminadas contienen variables relevantes desde el punto de vista analítico. Además, si los valores faltantes no se distribuyen de manera aleatoria, esta decisión puede introducir sesgos y empobrecer la capacidad explicativa o predictiva del conjunto de datos.


In [ ]:
df_3 = df.dropna(axis=1)
df_3

### 2.3 Imputación de valores faltantes

La imputación de valores faltantes consiste en reemplazar los datos ausentes por valores estimados a partir de la información disponible en el conjunto de datos. Esta estrategia permite conservar observaciones que, de otro modo, serían eliminadas, reduciendo la pérdida de información y favoreciendo la continuidad del análisis. La elección del método de imputación debe realizarse con cautela, ya que puede influir en la distribución de los datos y, en consecuencia, en la validez de los resultados obtenidos.

In [ ]:
df

### Comparar antes de elegir

Las celdas anteriores muestran *cómo* se imputa. Falta lo que la rúbrica evalúa:
**por qué se eligió ese método y no otro**. Ninguna estrategia es mejor en
abstracto; las cifras son las que deciden.

In [ ]:
# El cuaderno reutiliza la variable df en varias secciones. Para que este
# bloque sea independiente, se carga el conjunto en su propia variable.
df_stroke = pd.read_csv("data/raw/healthcare-dataset-stroke-data.csv")

def comparar_imputaciones(datos, columna, agrupador=None, n_grupos=4):
    """Compara el efecto de cuatro estrategias de imputación sobre una variable.

    Retorna
    -------
    pd.DataFrame con el efecto de cada estrategia sobre la distribución.
    """
    original = datos[columna]
    if original.isna().sum() == 0:
        raise ValueError(f"'{columna}' no tiene faltantes que comparar.")

    resultados = {
        "sin imputar (referencia)": original.dropna(),
        "eliminar filas": original.dropna(),
        "media": original.fillna(original.mean()),
        "mediana": original.fillna(original.median()),
    }
    if agrupador is not None:
        # qcut corta en tramos con la misma cantidad de casos; transform devuelve
        # una serie del mismo largo, con la mediana del grupo en cada fila.
        tramos = pd.qcut(datos[agrupador], q=n_grupos, duplicates="drop")
        resultados["mediana por grupo"] = original.fillna(
            original.groupby(tramos, observed=True).transform("median"))

    filas = [{
        "estrategia": nombre,
        "n": len(serie.dropna()),
        "media": round(serie.mean(), 2),
        "desv_est": round(serie.std(), 3),
        "asimetria": round(serie.skew(), 3),
    } for nombre, serie in resultados.items()]

    tabla = pd.DataFrame(filas)
    referencia = tabla.iloc[0]["desv_est"]
    # La columna decisiva: cuánto deforma cada método la dispersión original
    tabla["cambio_desv_%"] = ((tabla["desv_est"] - referencia) / referencia * 100).round(2)
    tabla["filas_perdidas"] = len(datos) - tabla["n"]
    return tabla


comparar_imputaciones(df_stroke, "bmi", agrupador="age")

**Cómo se lee.** `cambio_desv_%` es la columna que decide: toda imputación por
un valor central concentra los datos y **reduce artificialmente la desviación
estándar**. El método que menos la reduce es el que menos deforma la variable.
`filas_perdidas` muestra el costo de eliminar, y `asimetria` indica si la media
representa bien el centro.

> **Si las cuatro filas dan casi lo mismo**, la variable es simétrica y las
> estrategias son equivalentes: lo correcto es declararlo y elegir la más simple.
> Escribir «se descarta la media por la asimetría» cuando la tabla muestra
> asimetría nula es exactamente la incoherencia que la rúbrica penaliza.

Imputaremos por la media observable de la columna "Primera prueba" y "Segunda prueba"

In [ ]:
a = df['Primera prueba']
b = df['Segunda prueba']
prom_a = a.mean()
prom_b = b.mean()

# Una forma simple es usar un diccionario para guardar la información.
#La llaves del diccionario son los nombres de las variables (columnas) y el valor asociado es el promedio
diccionario = {'Primera prueba':prom_a, 'Segunda prueba': prom_b}

# El método .fillna sirve para reemplazar los valores NaN por los que queremos.
df2 = df.fillna(diccionario)
df2

Como se habra dado cuenta, el dataframe se ve modificado segun el valor promedio de la variable respectiva. Esto puede ser útil, a veces, pero otras no.

Por ejemplo, en la variable: "Segunda prueba", existe 3 MV. Esto es el 50% de los datos. Eso puede tener una repercusión fuerte en la distribución de los datos.

### 3. Codificación de variables categóricas

En ciencia de datos, muchas variables representan **categorías** y no cantidades numéricas. Estas se denominan **variables categóricas** y describen cualidades, clases o grupos, como por ejemplo `color`, `tipo de cliente` o `nivel de satisfacción`.

Dado que la mayoría de los algoritmos de aprendizaje automático requieren entradas numéricas, es necesario aplicar un proceso de **codificación** (*encoding*), es decir, transformar estas categorías en una representación numérica adecuada para el análisis y el modelamiento.

### ¿Qué es una variable categórica?

Una **variable categórica** es aquella cuyos valores representan clases o categorías, y no cantidades con significado aritmético. Dentro de este tipo de variables, es posible distinguir dos casos principales:

### a. Variables categóricas nominales

Las **variables nominales** son aquellas cuyas categorías **no presentan un orden natural** entre sí.

**Ejemplo:**
- color = {rojo, azul, verde}

En este caso, las categorías son diferentes, pero ninguna es superior o inferior a otra. Por ello, no es adecuado asignar códigos como:

- rojo = 1  
- azul = 2  
- verde = 3  

Esto podría llevar al modelo a interpretar erróneamente que existe una relación de orden entre las categorías.

#### ¿Qué técnica se utiliza?
Para este tipo de variables se emplea **One-Hot Encoding**, que transforma cada categoría en una columna binaria independiente.

**Ejemplo:**

- rojo → [1, 0, 0]  
- azul → [0, 1, 0]  
- verde → [0, 0, 1]  

De esta manera, se representa la pertenencia a una categoría sin imponer una jerarquía artificial.

### b. Variables categóricas ordinales

Las **variables ordinales** son aquellas cuyas categorías **sí poseen un orden o jerarquía natural**.

**Ejemplo:**
- nivel de satisfacción = {bajo, medio, alto}

Aquí sí existe una progresión lógica entre las categorías, por lo que es razonable asignar una codificación numérica que preserve ese orden.

#### ¿Qué técnica se utiliza?
Para este tipo de variables se utiliza **Label Encoding** o codificación ordinal.

**Ejemplo:**

- bajo = 0  
- medio = 1  
- alto = 2  

En este caso, los números no representan magnitudes exactas, pero sí mantienen la relación jerárquica entre las categorías.

### ¿Por qué se usa una técnica distinta en cada caso?

La elección del método de codificación depende de la naturaleza de la variable:

- Para variables **nominales**, se utiliza **One-Hot Encoding** porque evita crear relaciones de orden inexistentes.
- Para variables **ordinales**, se utiliza **Label Encoding** porque conserva la jerarquía natural entre las categorías.

Esta distinción es metodológicamente importante, ya que una codificación incorrecta puede afectar la forma en que el modelo interpreta los datos y, en consecuencia, deteriorar la calidad de los resultados.

### Resumen

- **Variable categórica**: representa clases o categorías.
- **Variable nominal**: no tiene orden natural.
- **Variable ordinal**: tiene una jerarquía entre sus categorías.
- **One-Hot Encoding**: se utiliza para variables nominales.
- **Label Encoding**: se utiliza para variables ordinales.

> **Nota:** En `scikit-learn`, `LabelEncoder` se utiliza principalmente para codificar la variable objetivo. Para variables ordinales de entrada suele preferirse `OrdinalEncoder`. Sin embargo, conceptualmente ambos responden a la misma idea: representar categorías mediante códigos numéricos respetando su orden.

### 3.1 Codificación de variables categóricas ordinales

En esta subsección estudiaremos el uso de Label Encoding, una técnica de codificación que permite transformar las categorías de una variable categórica ordinal en valores numéricos, preservando el orden inherente entre sus clases.

Para ilustrar este procedimiento, utilizaremos un conjunto de datos pequeño, lo que permitirá observar con mayor claridad cómo se realiza la transformación y cómo se representan los resultados.

In [ ]:
datos = {'alumno': ['Zutano', 'Mengano', 'Zutano','Pepe' ,'Fulanito, Cosme', 'Maria'],
         'Primera prueba': [7, 8,7,7,8.5, 10],
         'Segunda prueba': [10,9,10,4,10,6.7],
         'observaciones':  ['ninguna','libre','ninguna','ninguna','libre','oyente'],
         'estaturas':      ['alto','mediano','alto','bajo','bajo','alto'],
         'DNI':[23000000, 12389100,23000000, 99999, 1001,30406011]}
df = pd.DataFrame(datos)
df

In [ ]:
# Esta instrucción importa el módulo preprocessing de scikit-learn, que incluye herramientas para preparar y transformar datos antes del análisis o modelamiento.
from sklearn import preprocessing

# crear el objeto
le = preprocessing.LabelEncoder()

# definir manualmente el orden correcto
le.classes_ = np.array(['bajo', 'mediano', 'alto'])

# tomar la variable
data = df['estaturas']

# transformar
data_transformada = le.transform(data)

print(data_transformada)

In [ ]:
# ahora hacemos el cambio a los datos
df['estaturas'] = data_transformada
df

### Las tres formas de codificar una ordinal, comparadas

La celda anterior usa un codificador. Aquí se ve, ejecutándolo, por qué la
elección importa.

In [ ]:
# El cuaderno reutiliza la variable df en varias secciones. Para que este
# bloque sea independiente, se carga el conjunto en su propia variable.
df_stroke = pd.read_csv("data/raw/healthcare-dataset-stroke-data.csv")

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

orden_declarado = ["Unknown", "never smoked", "formerly smoked", "smokes"]
categorias = sorted(df_stroke["smoking_status"].unique())

# Opción A — LabelEncoder: numera según el orden ALFABÉTICO de las categorías
etiquetador = LabelEncoder().fit(df_stroke["smoking_status"])
codigos_label = {c: int(etiquetador.transform([c])[0]) for c in categorias}

# Opción B — OrdinalEncoder: respeta el orden que se le declara
codificador = OrdinalEncoder(categories=[orden_declarado], dtype=int)
codificador.fit(df_stroke[["smoking_status"]])
codigos_ordinal = {
    c: int(codificador.transform(pd.DataFrame({"smoking_status": [c]}))[0][0])
    for c in categorias
}

# Opción C — diccionario explícito
codigos_dict = {c: i for i, c in enumerate(orden_declarado)}

pd.DataFrame({
    "categoría": categorias,
    "A · LabelEncoder": [codigos_label[c] for c in categorias],
    "B · OrdinalEncoder": [codigos_ordinal[c] for c in categorias],
    "C · diccionario": [codigos_dict[c] for c in categorias],
}).sort_values("C · diccionario").reset_index(drop=True)

Mire la columna **A**. `LabelEncoder` no sabe nada del dominio: ordena
alfabéticamente, de modo que `formerly smoked` queda por debajo de
`never smoked`. La escala afirma que un exfumador tiene *menos* exposición al
tabaco que alguien que nunca fumó. Es falso, y ningún error se lanza.

> **El problema de fondo.** `LabelEncoder` está diseñado para codificar la
> **variable objetivo** de un problema de clasificación, donde los números son
> etiquetas sin orden. Usarlo sobre una variable predictora ordinal es un mal uso
> frecuente y silencioso.

In [ ]:
# Qué hace cada opción ante una categoría que nadie declaró
nueva = pd.DataFrame({"smoking_status": ["ocasional"]})

for nombre, fn in [("A · LabelEncoder ", lambda: etiquetador.transform(nueva["smoking_status"])),
                   ("B · OrdinalEncoder", lambda: codificador.transform(nueva))]:
    try:
        fn()
        print(f"{nombre}: la acepta sin avisar")
    except ValueError as e:
        print(f"{nombre}: ValueError — {str(e)[:52]}")

print(f"C · .map() a secas : devuelve "
      f"{nueva['smoking_status'].map(codigos_dict).iloc[0]} — falla en silencio")

Aquí gana la librería: los dos codificadores **fallan de inmediato**, mientras
que un `.map()` a secas devuelve `NaN` sin avisar, y ese `NaN` viaja por el resto
del pipeline hasta aparecer mucho después.

Por eso el diccionario explícito solo es preferible **si se valida antes de
mapear**:

```python
observadas = set(datos[columna].unique())
if not observadas.issubset(set(orden)):
    raise ValueError(f"Categorías no declaradas: {observadas - set(orden)}")
```

### 3.2 Codificación de variables categóricas nominales

En esta subsección estudiaremos el uso de One-Hot Encoding, una técnica de codificación que permite transformar las categorías de una variable categórica nominal en una representación numérica, sin imponer un orden artificial entre sus clases.

Para ilustrar este procedimiento, utilizaremos un conjunto de datos pequeño, lo que permitirá observar con mayor claridad cómo se realiza la transformación y cómo se representan los resultados.



In [ ]:
df

In [ ]:
le1 = preprocessing.LabelEncoder()
data1 = df['observaciones']
le1.fit(data1)
data_transformada1 = le1.transform(data1)
data_transformada1.reshape(-1,1)
df['observaciones'] = data_transformada1
ohe=preprocessing.OneHotEncoder()
d1 = data_transformada1.reshape(-1,1)
ohe.fit(d1)
data_nueva1 = ohe.transform(d1).toarray()
data_nueva1 = pd.DataFrame(data_nueva1)
df=pd.concat([df, data_nueva1], axis=1)
df

In [ ]:
# Obtener nombres correctos deseados
new_names = ['ninguna', 'libre', 'oyente']  # Orden debe coincidir con las categorías detectadas

# Crear diccionario para renombrar
rename_dict = {i: new_names[i] for i in range(len(new_names))}

# Renombrar columnas en el DataFrame
df.rename(columns=rename_dict, inplace=True)

df.head()

In [ ]:
df = df.drop(columns=['observaciones', 'alumno'], errors='ignore')

In [ ]:
df.head()

### 4. Normalización y estandarización

La normalización y la estandarización son técnicas de preprocesamiento orientadas a transformar variables numéricas para hacerlas comparables en escala y adecuadas para el modelamiento. La normalización reescala los valores a un intervalo acotado, usualmente entre 0 y 1, mientras que la estandarización centra los datos en torno a media 0 y desviación estándar 1. Su importancia radica en que numerosos algoritmos de aprendizaje automático son sensibles a la magnitud de las variables, de modo que diferencias de escala pueden sesgar el proceso de entrenamiento, afectar la convergencia numérica y distorsionar la contribución relativa de los atributos. En este sentido, ambas técnicas no constituyen un ajuste meramente operativo, sino una decisión metodológica relevante, ya que favorecen la estabilidad computacional, la interpretabilidad de los resultados y el desempeño de los modelos en contextos de ciencia de datos e inteligencia artificial.

In [ ]:
# Un boxplot muestra mediana, cuartiles y valores atípicos: es la forma más
# rápida de ver si las variables están en escalas comparables.
import matplotlib.pyplot as plt

columnas = ['Primera prueba', 'Segunda prueba', 'estaturas', 'DNI']
plt.figure(figsize=(8,5))
df[columnas].boxplot()
plt.title("Boxplot de variables seleccionadas")
plt.ylabel("Valores")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Algunas estadísticas básicas de solo las variables numéricas
df.describe()

In [ ]:
# Normalización y estandarización
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

4.1 Estandarización con standard scaler

La estandarización mediante StandardScaler transforma las variables numéricas de modo que queden centradas en media 0 y con desviación estándar 1. Este procedimiento es especialmente útil cuando las variables presentan escalas diferentes, ya que evita que aquellas con magnitudes mayores dominen el proceso de aprendizaje del modelo.

In [ ]:
# Se crea primero el objeto Standard Scaler
scaler = StandardScaler()

# Luego "entrenamos" con los datos para obtener los parámetros del escalamiento
scaler.fit(df)

# Finalmente aplicamos el escalamiento
array_scaler = scaler.transform(df)
array_scaler

### Se ve que entraga los datos más "desordenados". Esto pasa por que lo que entrega es un arreglo (array). Lo pasaremos a DataFrame

In [ ]:
df_scaler = pd.DataFrame(array_scaler)
df_scaler

**Ahora el problema es que las columnas perdieron sus nombres**

Podemos recuperar los nombres

In [ ]:
df_scaler.columns = df.columns
df_scaler

In [ ]:
# Mismo gráfico, ahora sobre los datos ESTANDARIZADOS: sirve para comparar
# el efecto del escalador contra el gráfico anterior.
import matplotlib.pyplot as plt

columnas = ['Primera prueba', 'Segunda prueba', 'estaturas', 'DNI']
plt.figure(figsize=(8,5))
df_scaler[columnas].boxplot()
plt.title("Boxplot de variables seleccionadas")
plt.ylabel("Valores")
plt.xticks(rotation=45)
plt.show()

Finalmente, corroboramos el resultado:

In [ ]:
df_scaler.describe()

4.2 Normalización con minmax scaler

La normalización mediante MinMaxScaler reescala las variables numéricas a un intervalo definido, usualmente entre 0 y 1. Este procedimiento resulta útil cuando las variables presentan magnitudes diferentes y se requiere llevarlas a una escala común para favorecer la estabilidad y el desempeño de ciertos modelos de aprendizaje automático.

In [ ]:
scaler = MinMaxScaler()
array_scaler = scaler.fit_transform(df)

df_minmax = pd.DataFrame(array_scaler, columns=df.columns)
df_minmax

In [ ]:
# Y ahora con MinMax: observe cómo el rango queda acotado entre 0 y 1, pero
# los valores extremos comprimen al resto de las observaciones.
columnas = ['Primera prueba', 'Segunda prueba', 'estaturas', 'DNI']
plt.figure(figsize=(8,5))
df_minmax[columnas].boxplot()
plt.title("Boxplot de variables seleccionadas")
plt.ylabel("Valores")
plt.xticks(rotation=45)
plt.show()

In [ ]:
df_minmax.describe()

4.3 Escalamiento robusto mediante RobustScaler

El escalamiento robusto mediante RobustScaler es una técnica de preprocesamiento orientada a transformar variables numéricas utilizando medidas de tendencia central y dispersión menos sensibles a valores extremos. En lugar de basarse en la media y la desviación estándar, este método emplea la mediana y el rango intercuartílico, lo que permite reducir la influencia de observaciones atípicas sobre la escala final de los datos. Su importancia radica en que, en numerosos problemas de ciencia de datos e inteligencia artificial, la presencia de outliers puede distorsionar significativamente otras estrategias de escalamiento, afectando la estabilidad numérica y el desempeño de los modelos. En este sentido, RobustScaler constituye una alternativa metodológicamente adecuada cuando se trabaja con distribuciones asimétricas o conjuntos de datos con alta sensibilidad a valores extremos.

In [ ]:
# RobustScaler centra en la MEDIANA y divide por el rango intercuartílico,
# en vez de usar media y desviación: por eso los valores extremos lo afectan menos.
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
array_scaler = scaler.fit_transform(df)

df_robust = pd.DataFrame(array_scaler, columns=df.columns)
df_robust

In [ ]:
# Tercer gráfico comparativo, con RobustScaler. Compare los tres para decidir
# cuál corresponde a la forma de sus datos.
columnas = ['Primera prueba', 'Segunda prueba', 'estaturas', 'DNI']
plt.figure(figsize=(8,5))
df_robust[columnas].boxplot()
plt.title("Boxplot de variables seleccionadas")
plt.ylabel("Valores")
plt.xticks(rotation=45)
plt.show()

### Los tres escaladores, comparados

Igual que con la imputación: no hay uno mejor, hay uno que corresponde a la forma
de sus datos.

| Escalador | Qué garantiza | Cuándo conviene | Riesgo |
| --- | --- | --- | --- |
| **StandardScaler** | Media 0, desviación 1 | Distribución simétrica | No acota el rango |
| **MinMaxScaler** | Rango [0, 1] | Se requiere rango acotado | Un extremo comprime todo el resto |
| **RobustScaler** | Mediana 0, RIC 1 | Hay valores extremos | Las cifras no quedan «redondas» |

In [ ]:
# El cuaderno reutiliza la variable df en varias secciones. Para que este
# bloque sea independiente, se carga el conjunto en su propia variable.
df_stroke = pd.read_csv("data/raw/healthcare-dataset-stroke-data.csv")

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Se escalan solo las continuas: las binarias ya están en una escala comparable.
numericas = ["age", "avg_glucose_level", "bmi"]
X = df_stroke[numericas].fillna(df_stroke[numericas].median())

resultados = {"original": X}
for nombre, clase in [("standard", StandardScaler),
                      ("minmax", MinMaxScaler),
                      ("robust", RobustScaler)]:
    # fit_transform aprende los estadísticos de X y aplica la transformación.
    # Devuelve un arreglo de NumPy sin nombres de columna: se reconstruye el
    # DataFrame para poder leer el resultado.
    resultados[nombre] = pd.DataFrame(
        clase().fit_transform(X), columns=numericas)

for nombre, valores in resultados.items():
    print(f"\n{nombre.upper()}")
    print(valores.describe().round(2).loc[["mean", "50%", "std", "min", "max"]])

In [ ]:
def verificar_escalamiento(valores, metodo, tol=1e-9):
    """Comprueba que el escalamiento produjo lo que el método promete."""
    if metodo == "standard":
        # ddof=0 pide la desviación POBLACIONAL, que es la que usa StandardScaler;
        # pandas usa ddof=1 por defecto y la comprobación no daría exactamente 1.
        assert np.allclose(valores.mean(), 0, atol=tol), "media distinta de 0"
        assert np.allclose(valores.std(ddof=0), 1, atol=tol), "desviación distinta de 1"
        return "media ≈ 0 y desviación ≈ 1"
    if metodo == "minmax":
        assert np.allclose(valores.min(), 0, atol=tol), "mínimo distinto de 0"
        assert np.allclose(valores.max(), 1, atol=tol), "máximo distinto de 1"
        return "todos los valores en el rango [0, 1]"
    if metodo == "robust":
        assert np.allclose(valores.median(), 0, atol=1e-8), "mediana distinta de 0"
        ric = valores.quantile(0.75) - valores.quantile(0.25)
        assert np.allclose(ric, 1, atol=1e-8), "rango intercuartílico distinto de 1"
        return "mediana ≈ 0 y rango intercuartílico ≈ 1"
    raise ValueError(f"Método desconocido: {metodo}")


for metodo in ["standard", "minmax", "robust"]:
    print(f"[OK] {metodo:<9} {verificar_escalamiento(resultados[metodo]['bmi'], metodo)}")

In [ ]:
import matplotlib.pyplot as plt

# subplots(1, 4) crea una figura con cuatro paneles en una fila
fig, ejes = plt.subplots(1, 4, figsize=(15, 3.5))

# zip recorre en paralelo los paneles y los resultados: cada escalador va al suyo
for eje, (nombre, valores) in zip(ejes, resultados.items()):
    valores.boxplot(ax=eje)
    eje.set_title(nombre.capitalize())
    eje.tick_params(axis="x", rotation=30, labelsize=8)

plt.suptitle("Efecto de cada escalador sobre las mismas variables")
plt.tight_layout()
plt.show()

> **Coherencia entre decisiones.** Si eligió la mediana para imputar *porque hay
> valores extremos*, la misma evidencia debería llevarlo a `RobustScaler`. Elegir
> mediana para imputar y `StandardScaler` para escalar no está prohibido, pero
> exige explicar por qué el mismo hecho lleva a criterios distintos. Eso separa un
> pipeline pensado de una sucesión de pasos.

> **Atención — fuga de datos.** Aquí los estadísticos se calculan sobre todo el
> conjunto porque no hay partición. Cuando la haya, tanto el valor de imputación
> como el escalador deben ajustarse **solo con el conjunto de entrenamiento**.

## Ejercicio — sección XII

Escriba `preparar(ruta)` que ejecute el pipeline completo y devuelva
`(datos_procesados, bitacora)`, donde la bitácora es un diccionario con lo que
hizo. La función debe:

1. Cargar el archivo.
2. Descartar la columna identificadora.
3. Imputar `bmi` con la **mediana** y registrar cuántos valores imputó.
4. Codificar `smoking_status` con el orden declarado.
5. Devolver los datos y la bitácora.

In [ ]:
def preparar(ruta):
    """TODO: cargar, limpiar, codificar y devolver (datos, bitacora)."""
    pass


ORDEN = ["Unknown", "never smoked", "formerly smoked", "smokes"]
datos_listos, bitacora = preparar("data/raw/healthcare-dataset-stroke-data.csv")

for clave, valor in bitacora.items():
    print(f"  {clave}: {valor}")

assert "id" not in datos_listos.columns, "El identificador debía descartarse"
assert datos_listos["bmi"].isna().sum() == 0, "Quedaron faltantes en bmi"
assert "smoking_ord" in datos_listos.columns, "Falta la variable ordinal codificada"
assert datos_listos["smoking_ord"].max() == 3
assert bitacora["imputados_bmi"] > 0
print("\nCorrecto")

<details>
<summary><b>Ver solución</b></summary>

```python
def preparar(ruta):
    """Ejecuta el pipeline y devuelve los datos junto con su bitácora."""
    orden = ["Unknown", "never smoked", "formerly smoked", "smokes"]
    bitacora = {}

    datos = pd.read_csv(ruta)
    bitacora["filas_cargadas"] = len(datos)

    datos = datos.drop(columns=["id"])

    n_faltantes = int(datos["bmi"].isna().sum())
    mediana = datos["bmi"].median()
    datos["bmi"] = datos["bmi"].fillna(mediana)
    bitacora["imputados_bmi"] = n_faltantes
    bitacora["valor_imputado"] = round(mediana, 2)

    mapa = {c: i for i, c in enumerate(orden)}
    datos["smoking_ord"] = datos["smoking_status"].map(mapa)
    bitacora["orden_declarado"] = mapa

    return datos, bitacora
```

</details>

# XIII. Ejercicios

## Fase 1

### Extensión de Fase 1 — Uso de Librerías Científicas

En la primera parte del cuaderno implementamos manualmente:

Lectura de CSV

Separación por diagnóstico

Cálculo de promedios

Detección de valores nulos

Función de decisión heurística

Ahora realizaremos exactamente el mismo análisis, pero utilizando librerías especializadas como:

pandas

numpy

El objetivo es comparar:

✔ Claridad del código ✔ Reducción de complejidad ✔ Eficiencia computacional ✔ Expresividad del lenguaje

Trabajaremos con el mismo archivo: healthcare-dataset-stroke-data.csv

In [ ]:
# -----------------------------------------
# IMPORTACIÓN DE LIBRERÍAS
# -----------------------------------------

import pandas as pd      # Manejo estructurado de datos
import numpy as np       # Operaciones numéricas eficientes

# Verificamos que las librerías están disponibles
print("Librerías cargadas correctamente.")

1. Lectura del Dataset con Pandas

pd.read_csv(), que:

Detecta automáticamente encabezados
Maneja valores faltantes (NaN)
Construye una estructura tabular eficiente

In [ ]:
df = pd.read_csv("data/raw/healthcare-dataset-stroke-data.csv")

# Mostrar primeras filas
print("Primeros registros:")
print(df.head())

# Dimensiones del dataset
print("\nDimensiones:", df.shape)

## 0. Antes de tocar nada: el diccionario de variables

Este es el paso que decide todo lo que viene después, y el producto documental
que la **Fase 1** pide entregar.

La idea central: **el tipo de dato no dice qué es una variable**. Tres columnas
`int64` pueden ser un identificador, un conteo y un sí/no, y cada una exige un
tratamiento distinto. Lo que determina el preprocesamiento es el **rol
analítico**, y ese lo declara usted, no la librería.

| Rol analítico | Qué preprocesamiento exige |
| --- | --- |
| **Continua** | Detección de atípicos, escalamiento, imputación |
| **Discreta** | Decidir si se trata como número o como categoría |
| **Binaria** | Codificación 0/1 y revisión de desbalance |
| **Nominal** | One-hot encoding |
| **Ordinal** | Codificación con el orden declarado explícitamente |
| **Alta cardinalidad** | Agrupación o codificación por frecuencia |
| **Identificador** | Verificar unicidad y excluir del análisis |

In [ ]:
# El cuaderno reutiliza la variable df en varias secciones. Para que este
# bloque sea independiente, se carga el conjunto en su propia variable.
df_stroke = pd.read_csv("data/raw/healthcare-dataset-stroke-data.csv")

# El diccionario se DECLARA (lo que sabemos del dominio) y luego se COMPLETA
# con lo observado en el archivo: no se afirma nada que no se verifique.
DECLARADO = [
    ("id",                "identificador", "Identificador único del paciente"),
    ("gender",            "nominal",       "Female, Male, Other"),
    ("age",               "continua",      "Edad en años"),
    ("hypertension",      "binaria",       "1 si presenta hipertensión"),
    ("heart_disease",     "binaria",       "1 si presenta cardiopatía"),
    ("ever_married",      "binaria",       "Yes / No"),
    ("work_type",         "nominal",       "Tipo de ocupación"),
    ("Residence_type",    "binaria",       "Urban / Rural"),
    ("avg_glucose_level", "continua",      "Glucosa promedio en sangre"),
    ("bmi",               "continua",      "Índice de masa corporal"),
    ("smoking_status",    "ordinal",       "Unknown < never < formerly < smokes"),
    ("stroke",            "objetivo",      "1 si el paciente presentó el evento"),
]

diccionario = pd.DataFrame(DECLARADO, columns=["variable", "rol", "descripcion"])

# Columnas observadas: se leen del propio archivo, no se escriben a mano
diccionario["dtype"] = [str(df_stroke[v].dtype) for v in diccionario["variable"]]
diccionario["n_unicos"] = [int(df_stroke[v].nunique()) for v in diccionario["variable"]]
diccionario["pct_nulos"] = [round(df_stroke[v].isna().mean() * 100, 1)
                            for v in diccionario["variable"]]
diccionario

In [ ]:
# El diccionario es un entregable de la Fase 1, no una tabla de paso: se guarda.
Path("docs").mkdir(exist_ok=True)
diccionario.to_csv("docs/diccionario_variables.csv", index=False)

print("Composición por rol analítico:")
for rol, n in diccionario["rol"].value_counts().items():
    print(f"  {rol:<15} {n}")

# Las listas que usa el resto de la sección salen del diccionario, no de nombres
# escritos a mano: si el diccionario cambia, el preprocesamiento se adapta solo.
continuas = diccionario.loc[diccionario["rol"] == "continua", "variable"].tolist()
nominales = diccionario.loc[diccionario["rol"] == "nominal", "variable"].tolist()
print(f"\nContinuas: {continuas}")
print(f"Nominales: {nominales}")

> **Por qué esto es la base de todo.** Las decisiones que siguen —faltantes,
> categóricas, escalamiento— se aplican según el rol de cada variable. Si el
> diccionario está mal, el preprocesamiento estará mal aunque el código sea
> impecable.

In [ ]:
df

2. Detección de Valores Nulos
En la versión manual tuvimos que recorrer cada fila.

Con pandas podemos hacerlo directamente usando:

isnull()sum()

In [ ]:
# -----------------------------------------
# DETECCIÓN DE NULOS
# -----------------------------------------

valores_nulos = df.isnull().sum()

print("Valores faltantes por columna:")
print(valores_nulos)

3. Estadísticas Descriptivas
Anteriormente calculamos promedio y varianza manualmente.

Con pandas podemos obtener:

count

mean


std

min

max

percentiles

In [ ]:
# -----------------------------------------
# ESTADÍSTICAS DESCRIPTIVAS
# -----------------------------------------

estadisticas = df.describe()

print("Resumen estadístico:")
print(estadisticas)

In [ ]:
df.describe()

4. Separación por Outcome

En la versión manual utilizamos listas.

Con pandas podemos usar filtrado booleano.

In [ ]:
# -----------------------------------------
# SEPARACIÓN POR DIAGNÓSTICO
# -----------------------------------------

grupo_stroke = df[df["stroke"] == 1]
grupo_no_stroke = df[df["stroke"] == 0]

print("Pacientes con stroke:", len(grupo_stroke))
print("Pacientes sin stroke:", len(grupo_no_stroke))

5. Promedios por Grupo

En la implementación manual usamos un ciclo for.

Con pandas podemos usar:

groupby()

In [ ]:
promedios = df.groupby("stroke")[["avg_glucose_level", "bmi", "age"]].mean()

print("Promedios por grupo:")
print(promedios)

## Fase 2

Fase 2 — Obtención, Limpieza y Transformación de Datos
En esta fase construiremos un pipeline funcional completo que permita:

Leer el archivo Diabetes_Data_Extended.csv.
Detectar valores nulos.
Detectar outliers simples.
Limpiar datos.
Transformar variables.
Validar resultados intermedios.
Preparar datos para análisis posteriores.
Todo se implementará primero sin librerías externas.

Objetivos de la Fase 2
Al finalizar esta fase el estudiante será capaz de:

Implementar un pipeline funcional en Python.
Diseñar funciones con responsabilidades claras.
Realizar limpieza sistemática de datos.
Validar resultados intermedios del procesamiento.
Preparar los datos para análisis posteriores.
Configuración del Entorno
El entorno requerido corresponde únicamente a una instalación estándar de Python 3.

Ejercicio
Explique por qué utilizar únicamente Python estándar favorece la portabilidad.

In [ ]:
# -----------------------------------------
# MÓDULOS ESTÁNDAR PERMITIDOS
# -----------------------------------------
# En esta sección importamos únicamente librerías
# incluidas en la distribución estándar de Python.
#
# Esto garantiza:
# ✔ Portabilidad
# ✔ Reproducibilidad
# ✔ Independencia de librerías externas
# ✔ Compatibilidad con cualquier entorno Python 3


import os
# Permite interactuar con el sistema operativo.
# Útil para:
# - Verificar existencia de archivos
# - Manejar rutas
# - Crear directorios
# - Automatizar ejecución

import time
# Permite medir tiempos de ejecución.
# Fundamental para:
# - Evaluar eficiencia del pipeline
# - Comparar versiones del algoritmo
# - Estimar complejidad empírica

import math
# Proporciona funciones matemáticas avanzadas.
# Se utiliza en:
# - Cálculo de correlación (raíz cuadrada)
# - Operaciones numéricas precisas
# - Fórmulas estadísticas

import copy
# Permite realizar copias profundas (deepcopy).
# Es importante cuando:
# - Queremos transformar datos
# - Sin modificar el dataset original
# - Manteniendo integridad estructural

print("Entorno configurado correctamente.")

PARTE 1 — IMPLEMENTACIÓN MANUAL
Obtención y Exploración de Datos
Exploración manual del dataset para comprender su estructura.

In [ ]:
# -----------------------------------------
# FUNCIÓN: cargar_csv
# -----------------------------------------
# Esta función representa la etapa de INGESTA DE DATOS.
#
# Objetivo:
# - Leer manualmente un archivo CSV.
# - Convertir cada fila en un diccionario.
# - Manejar valores faltantes como None.
#
# Importante:
# No usamos pandas. Todo se hace manualmente
# para comprender cómo funciona internamente
# la lectura estructurada de datos.


def cargar_csv(ruta):

    """
    Lee manualmente un archivo CSV.
    Maneja valores vacíos como None.

    Parámetro:
    ruta (str): Ruta del archivo CSV.

    Retorna:
    encabezados (list): Lista de nombres de columnas.
    datos (list): Lista de diccionarios (registros).
    """

    # Abrimos el archivo en modo lectura
    with open(ruta, "r", encoding="utf-8") as archivo:
        lineas = archivo.readlines()

    # Primera línea → nombres de columnas
    encabezados = lineas[0].strip().split(",")

    # Lista que almacenará todos los registros
    datos = []

    # Definimos columnas numéricas del dataset clínico
    columnas_numericas = [
        "id", "age", "hypertension", "heart_disease",
        "avg_glucose_level", "bmi", "stroke"
    ]

    # Recorremos desde la segunda línea (datos reales)
    for linea in lineas[1:]:

        # Separar valores por coma
        valores = linea.strip().split(",")

        # Diccionario para representar un paciente
        fila = {}

        # Asociamos cada valor con su columna
        for i in range(len(encabezados)):

            valor = valores[i]
            columna = encabezados[i]

            # Manejo básico de valores faltantes
            # Si el valor está vacío o es "nan"
            # lo convertimos en None
            if valor == "" or valor.lower() == "nan":
                fila[columna] = None

            # Convertimos a float SOLO si es numérico
            elif columna in columnas_numericas:
                try:
                    fila[columna] = float(valor)
                except:
                    fila[columna] = None

            # Variables categóricas se mantienen como texto
            else:
                fila[columna] = valor

        # Agregamos el registro a la lista principal
        datos.append(fila)

    return encabezados, datos


# -----------------------------------------
# EJECUCIÓN DE LA FUNCIÓN
# -----------------------------------------
# Aquí comienza formalmente el pipeline.
# Cargamos el dataset clínico de stroke.

encabezados, datos = cargar_csv("data/raw/healthcare-dataset-stroke-data.csv")

print("Registros cargados:", len(datos))
print("Columnas:", encabezados)

Ejercicio 2
Identifique qué columnas podrían contener valores faltantes y por qué.

Detección de Nulos

In [ ]:
# -----------------------------------------
# FUNCIÓN: contar_nulos
# -----------------------------------------
# Esta función pertenece a la etapa de
# VALIDACIÓN Y LIMPIEZA DE DATOS.
#
# Objetivo:
# - Detectar cuántos valores faltantes (None)
#   existen por cada columna.
#
# ¿Por qué es importante?
# - Permite evaluar calidad del dataset.
# - Es un paso previo obligatorio antes
#   de análisis estadístico o modelado.
# - Ayuda a decidir estrategias de imputación
#   o eliminación de registros.


def contar_nulos(datos):
    # Diccionario donde almacenaremos el conteo
    # Ejemplo esperado:
    # {"bmi": 10, "avg_glucose_level": 2}
    conteo = {}

    # Recorremos cada registro del dataset
    # Complejidad: O(n)
    for fila in datos:

        # Recorremos cada variable del registro
        # Complejidad interna: O(m)
        for clave, valor in fila.items():

            # Si el valor es None (faltante)
            if valor is None:

                # Si la columna aún no está en el diccionario
                # la inicializamos en 0
                if clave not in conteo:
                    conteo[clave] = 0

                # Incrementamos el contador
                conteo[clave] += 1

    return conteo


# -----------------------------------------
# EJECUCIÓN DE VALIDACIÓN
# -----------------------------------------
# Mostramos el resumen de valores faltantes
# detectados en el dataset clínico.

print("Valores faltantes:", contar_nulos(datos))

In [ ]:
# .items() recorre el diccionario entregando (clave, valor) en cada vuelta.
# El formato :.2% convierte la proporción en porcentaje con dos decimales.
nulos = contar_nulos(datos)
total = len(datos)

print("\nProporción de valores faltantes:")
for col, val in nulos.items():
    print(f"{col}: {val} ({val/total:.2%})")

Limpieza de Datos

Se eliminan registros con valores no válidos.

In [ ]:
# -----------------------------------------
# FUNCIÓN: limpiar_datos
# -----------------------------------------
# Esta función corresponde a la etapa de
# LIMPIEZA FORMAL DEL DATASET.
#
# Objetivo:
# - Eliminar registros con datos inválidos
#   en variables críticas (avg_glucose_level y bmi).
#
# ¿Por qué estas variables?
# - Son variables centrales en el análisis.
# - Se utilizan en la función de decisión heurística.
# - Valores <= 0 no tienen sentido clínico.
#
# Estrategia aplicada:
# - Si hay None → descartar registro.
# - Si valor <= 0 → descartar registro.
#
# No se modifica el dataset original directamente.
# Se crea una nueva lista limpia.


def limpiar_datos(datos):
    """
    Elimina registros con valores inválidos básicos.

    Parámetro:
    datos (list): Dataset original.

    Retorna:
    list: Dataset filtrado.
    """

    # Nueva lista que almacenará solo registros válidos
    datos_limpios = []

    # Recorremos todos los registros
    # Complejidad: O(n)
    for fila in datos:

        #Eliminación por valores faltantes
        if fila["avg_glucose_level"] is None or fila["bmi"] is None:
            continue

        # Eliminación por valores físicamente inválidos
        if fila["avg_glucose_level"] <= 0 or fila["bmi"] <= 0:
            continue

        # Si pasa todas las validaciones, se conserva
        datos_limpios.append(fila)

    return datos_limpios


# -----------------------------------------
# EJECUCIÓN DE LA LIMPIEZA
# -----------------------------------------

datos = limpiar_datos(datos)

print("Registros tras limpieza:", len(datos))

Ejercicio 3

¿Por qué no siempre conviene eliminar todas las filas con valores faltantes?

Separación por Diagnóstico

In [ ]:
# -----------------------------------------
# FUNCIÓN: separar_por_diagnostico
# -----------------------------------------
# Esta función pertenece a la etapa de
# TRANSFORMACIÓN Y PREPARACIÓN PARA ANÁLISIS SUPERVISADO.
#
# Objetivo:
# - Dividir el dataset en dos subconjuntos:
#     1 → Pacientes con stroke
#     0 → Pacientes sin stroke
#
# ¿Por qué es importante?
# - Permite comparar estadísticas entre grupos.
# - Es la base del análisis supervisado.
# - Facilita cálculo de métricas por clase.
#
# No modifica el dataset original.
# Solo genera dos nuevas estructuras.


def separar_por_diagnostico(datos):
    # Lista para pacientes con diagnóstico positivo
    grupo_stroke = []

    # Lista para pacientes sin diagnóstico
    grupo_no_stroke = []

    # Recorremos todo el dataset
    # Complejidad: O(n)
    for fila in datos:

        # Clasificación según variable objetivo
        if fila["stroke"] == 1:
            grupo_stroke.append(fila)
        else:
            grupo_no_stroke.append(fila)

    # Retornamos ambos grupos
    return grupo_stroke, grupo_no_stroke


# -----------------------------------------
# EJECUCIÓN DE LA SEPARACIÓN
# -----------------------------------------

grupo_stroke, grupo_no_stroke = separar_por_diagnostico(datos)

print("Stroke:", len(grupo_stroke))
print("No Stroke:", len(grupo_no_stroke))

In [ ]:
print("Total:", len(datos))
print("Verificación:", len(grupo_stroke) + len(grupo_no_stroke))

Cálculo de Promedios

In [ ]:
# -----------------------------------------
# FUNCIÓN: calcular_promedios
# -----------------------------------------
# Esta función pertenece a la etapa de
# ANÁLISIS DESCRIPTIVO DEL DATASET.
#
# Objetivo:
# - Calcular el promedio de múltiples variables
#   dentro de un grupo específico (por ejemplo:
#   pacientes con o sin stroke).
#
# Características importantes:
# - Ignora valores None (faltantes).
# - Evita división por cero.
# - Devuelve resultados estructurados en diccionario.
#
# Esto permite comparar estadísticamente grupos
# antes de aplicar modelos más complejos.


def calcular_promedios(grupo, variables):
    # Diccionario donde almacenaremos resultados
    resultados = {}

    # Recorremos cada variable solicitada
    # Complejidad externa: O(k)
    for var in variables:

        suma = 0
        contador = 0

        # Recorremos todos los registros del grupo
        # Complejidad interna: O(n)
        for fila in grupo:
            valor = fila[var]

            # Ignoramos valores faltantes y aseguramos tipo numérico
            if valor is not None and isinstance(valor, (int, float)):
                suma += valor
                contador += 1

        # Evitamos división por cero
        resultados[var] = suma / contador if contador > 0 else 0

    return resultados


# -----------------------------------------
# EJECUCIÓN DEL ANÁLISIS DESCRIPTIVO
# -----------------------------------------

variables = ["avg_glucose_level", "bmi", "age"]

# Calculamos promedios por grupo
prom_stroke = calcular_promedios(grupo_stroke, variables)
prom_no_stroke = calcular_promedios(grupo_no_stroke, variables)

print("Promedios Stroke:", prom_stroke)
print("Promedios No Stroke:", prom_no_stroke)

In [ ]:
# Diferencias entre grupos
diff = {var: prom_stroke[var] - prom_no_stroke[var] for var in variables}

print("Diferencias (Stroke - No Stroke):", diff)

Detección de Outliers Simples

In [ ]:
# -----------------------------------------
# FUNCIÓN: detectar_outliers
# -----------------------------------------
# Esta función pertenece a la etapa de
# VALIDACIÓN AVANZADA DEL DATASET.
#
# Objetivo:
# - Detectar registros con valores extremos
#   según reglas simples basadas en umbrales.
#
# Regla aplicada:
# - avg_glucose_level > 250
# - bmi > 60
#
# ⚠ Importante:
# - No elimina los registros.
# - Solo los identifica.
# - Permite análisis posterior.
#
# Este enfoque es determinístico y simple.
# No utiliza métodos estadísticos como:
# - IQR
# - Z-score
# - Desviación estándar
#
# Complejidad: O(n)


def detectar_outliers(datos):
    # Lista donde almacenaremos registros sospechosos
    outliers = []

    # Recorremos todos los registros
    for fila in datos:

        # Evaluamos condiciones extremas (validando None)
        if (
            (fila["avg_glucose_level"] is not None and fila["avg_glucose_level"] > 250)
            or
            (fila["bmi"] is not None and fila["bmi"] > 60)
        ):
            outliers.append(fila)

    return outliers


# -----------------------------------------
# EJECUCIÓN DE DETECCIÓN
# -----------------------------------------

print("Outliers detectados:", len(detectar_outliers(datos)))

In [ ]:
outliers = detectar_outliers(datos)

print("Ejemplos de outliers:", outliers[:3])

Transformación (Escalamiento)

In [ ]:
# -----------------------------------------
# FUNCIÓN: escalar_glucose
# -----------------------------------------
# Esta función pertenece a la etapa de
# TRANSFORMACIÓN Y PREPROCESAMIENTO.
#
# Objetivo:
# - Escalar la variable "avg_glucose_level"
#   dividiéndola por 200.
#
# ¿Por qué 200?
# - Es un valor de referencia aproximado
#   para llevar la variable a un rango cercano a [0,1].
# - Permite reducir magnitudes grandes.
# - Facilita comparación entre variables.
#
# Importante:
# - No modifica el dataset original.
# - Crea una nueva estructura.
# - Aplica copia superficial del diccionario.
#
# Complejidad: O(n)


def escalar_glucose(datos):
    # Nueva lista donde almacenaremos los registros transformados
    datos_nuevos = []

    # Recorremos todos los registros
    for fila in datos:

        # Creamos copia del registro
        # para no modificar el original
        nueva = fila.copy()

        # Aplicamos transformación simple (validando None)
        if fila["avg_glucose_level"] is not None:
            nueva["avg_glucose_level"] = fila["avg_glucose_level"] / 200
        else:
            nueva["avg_glucose_level"] = None

        # Guardamos el registro transformado
        datos_nuevos.append(nueva)

    return datos_nuevos


# -----------------------------------------
# EJECUCIÓN DE TRANSFORMACIÓN
# -----------------------------------------

datos_escalados = escalar_glucose(datos)

# Validación estructural:
# Verificamos que no se hayan perdido registros
assert len(datos) == len(datos_escalados)

print("Escalamiento aplicado correctamente.")


Validación con Condiciones

In [ ]:
# -----------------------------------------
# BLOQUE DE VALIDACIÓN DEL PIPELINE
# -----------------------------------------
# Esta sección corresponde a la etapa de
# VALIDACIÓN ESTRUCTURAL DEL SISTEMA.
#
# Objetivo:
# - Verificar que cada etapa anterior
#   se ejecutó correctamente.
# - Detectar errores antes de continuar.
# - Asegurar consistencia del flujo completo.
#
# Se definen condiciones de validación para:
# ✔ Hacer explícita cada verificación
# ✔ Mejorar legibilidad
# ✔ Facilitar debugging
# ✔ Documentar supuestos del sistema


# Verifica que el dataset no esté vacío
cond_datos_cargados = len(datos) > 0

# Verifica que la columna crítica exista
cond_columnas_validas = "avg_glucose_level" in encabezados

# Verifica que la separación por diagnóstico
# conserve el número total de registros
cond_separacion_correcta = len(grupo_stroke) + len(grupo_no_stroke) == len(datos)


# Assertions → detienen el programa si algo falla
assert cond_datos_cargados
assert cond_columnas_validas
assert cond_separacion_correcta


print("Validaciones superadas correctamente.")

In [ ]:
assert len(datos) > 0, "Dataset vacío"
assert "avg_glucose_level" in encabezados, "Columna faltante"
assert len(grupo_stroke) + len(grupo_no_stroke) == len(datos), "Error en separación"

Medición de Tiempo

In [ ]:
# -----------------------------------------
# MEDICIÓN DE TIEMPO DEL PIPELINE
# -----------------------------------------
# Esta sección pertenece a la etapa de
# EVALUACIÓN DE EFICIENCIA.
#
# Objetivo:
# - Medir cuánto tarda la carga y limpieza.
# - Obtener evidencia empírica del rendimiento.
# - Relacionar resultados con análisis de complejidad O(n).
#
# time.time() devuelve el tiempo actual en segundos
# desde el epoch (1 de enero de 1970).


# Guardamos instante inicial
inicio = time.time()


# Ejecutamos etapas críticas del pipeline
# 1) Carga del dataset clínico
encabezados, datos_temp = cargar_csv("data/raw/healthcare-dataset-stroke-data.csv")

# 2) Limpieza básica
datos_temp = limpiar_datos(datos_temp)


# Guardamos instante final
fin = time.time()


# Diferencia de tiempos = duración total
print("Tiempo de ejecución:", fin - inicio, "segundos")

In [ ]:
duracion = fin - inicio
print(f"Tiempo total del pipeline: {duracion:.6f} segundos")

VERSIÓN CON PANDAS Y NUMPY
Anexo — Implementación con Pandas y NumPy
Ahora repetiremos el mismo análisis usando:

pandas
numpy
El objetivo es comparar:

Claridad
Cantidad de código
Eficiencia
Complejidad cognitiva


Lectura con Pandas

In [ ]:
# -----------------------------------------
# LECTURA CON PANDAS
# -----------------------------------------
# Aquí comenzamos el ANEXO VECTORIZADO.
#
# Objetivo:
# - Mostrar cómo librerías especializadas
# simplifican el mismo pipeline.
#
# Diferencia clave:
# - pandas implementa estructuras tipo DataFrame
# - Las operaciones están optimizadas en C
# - Se reduce código explícito
# - Se mejora legibilidad

import pandas as pd
import numpy as np

# Lectura directa del CSV (sin loops manuales)
df = pd.read_csv("data/raw/healthcare-dataset-stroke-data.csv")

# shape devuelve (filas, columnas)
print("Dimensiones:", df.shape)

Limpieza con Pandas

In [ ]:
# -----------------------------------------
# LIMPIEZA CON PANDAS
# -----------------------------------------
# Equivalente a nuestra función limpiar_datos(),
# pero ahora usando operaciones vectorizadas.
#
# dropna elimina registros con valores nulos
# en columnas específicas.
#
# Luego aplicamos filtro booleano en una sola línea.

df_limpio = df.dropna(subset=["avg_glucose_level", "bmi"])

df_limpio = df_limpio[
    (df_limpio["avg_glucose_level"] > 0) &
    (df_limpio["bmi"] > 0)
]

print("Registros tras limpieza:", len(df_limpio))

Promedios con groupby

In [ ]:
# -----------------------------------------
# PROMEDIOS CON GROUPBY
# -----------------------------------------
# Equivalente a calcular_promedios()
# pero usando agrupamiento vectorizado.
#
# groupby("stroke") separa automáticamente
# los grupos 0 y 1.
#
# mean() calcula promedio sin loops explícitos.

variables = ["avg_glucose_level", "bmi", "age"]

promedios = df_limpio.groupby("stroke")[variables].mean()

print(promedios)

Detección de Outliers con IQR

In [ ]:
# -----------------------------------------
# DETECCIÓN DE OUTLIERS CON IQR
# -----------------------------------------
# Ahora utilizamos un método estadístico formal.
#
# IQR (Interquartile Range):
# IQR = Q3 - Q1
#
# Regla:
# Outlier si:
# x < Q1 - 1.5*IQR
# x > Q3 + 1.5*IQR
#
# Esto es más robusto que usar umbrales fijos.

Q1 = df_limpio["avg_glucose_level"].quantile(0.25)
Q3 = df_limpio["avg_glucose_level"].quantile(0.75)

IQR = Q3 - Q1

lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

outliers = df_limpio[
    (df_limpio["avg_glucose_level"] < lim_inf) |
    (df_limpio["avg_glucose_level"] > lim_sup)
]

print("Outliers detectados:", len(outliers))

Comparación Final
Comparación Final
Comparación Manual vs Librerías
Aspecto	Manual	Pandas/NumPy
Lectura	30+ líneas	1 línea
Limpieza	Bucles	dropna + filtros
Promedios	Ciclos anidados	groupby().mean()
Outliers	Condicional	IQR vectorizado


---

# XIV. Autoevaluación

Si responde estas preguntas sin volver atrás, está en condiciones de abordar el
avance de las Fases 1 y 2.

| # | Pregunta | Sección |
| --- | --- | --- |
| 1 | ¿Por qué `lista_b = lista_a` no crea una copia, y cómo se copia realmente? | V |
| 2 | ¿Qué diferencia hay entre `==` e `is`, y cuál se usa con `None`? | IV |
| 3 | ¿Por qué `0.1 + 0.2 == 0.3` es `False`, y cómo se comparan decimales? | II |
| 4 | ¿Qué distingue el `if` antes del `for` del `if` después, en una comprensión? | VI-B |
| 5 | ¿Por qué un argumento por defecto mutable es peligroso? | III |
| 6 | ¿Qué hace `axis=0` frente a `axis=1` en una operación de pandas? | VIII |
| 7 | ¿Por qué `LabelEncoder` no sirve para una variable ordinal? | IX |
| 8 | ¿Qué le hace a la desviación estándar imputar por la mediana? | IX |
| 9 | ¿Qué archivos NO deben entrar al repositorio, y por qué? | VI-C |
| 10 | ¿Qué significa que un notebook corra con *Restart & Run All*? | 0 |

---

# XV. Del cuaderno al proyecto

Nada de lo practicado aquí es teoría suelta: todo reaparece en el avance que se
entrega al cierre de la semana.

| Sección de este cuaderno | Dónde se aplica | Criterio de la rúbrica |
| --- | --- | --- |
| III. Funciones · VI. Módulos | Cada etapa del pipeline encapsulada | Codificación funcional (9 pts) |
| VII. Excepciones | Carga de archivos y validación de categorías | Validación técnica (6 pts) |
| VI-C. Entorno y versiones | Entorno virtual, `requirements.txt`, repositorio | Reproducibilidad (6 pts) · Control de versiones (6 pts) |
| VIII. NumPy y pandas | Todo el pipeline de la Fase 2 | Aplicación de herramientas (9 pts) |
| IX. Preprocesamiento | Limpieza, codificación y escalamiento justificados | Preprocesamiento (6 pts) |
| IX.0. Diccionario de variables | Documentación del conjunto de datos | Documentación del proceso (6 pts) |

**La regla que atraviesa todo el cuaderno:** primero medir, después decidir, y
justificar con la medición. Si escribe «hay valores extremos», la tabla tiene que
mostrarlos. Afirmar sin evidencia es lo que más penaliza la rúbrica.

> **Antes de pasar al proyecto**, ejecute este cuaderno completo con
> *Kernel → Restart Kernel and Run All Cells*. Si falla, el error está aquí y no
> en su entrega — que es exactamente el mejor lugar para encontrarlo.